In [1]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date, datetime, timedelta
from itertools import product
from copy import deepcopy
from gen_variable_standard_static import metrics_search_for_two_fragments_df
from tqdm import tqdm

In [2]:
systems_cleaned = pd.read_csv('../../../data/core/systems_cleaned.csv')
systems_cleaned.columns

Index(['system_id', 'system_public_name', 'site_location',
       'timezone_or_utc_offset', 'latitude', 'longitude', 'elevation_m',
       'dc_capacity_kW', 'kg_climate', 'pvcz_composite', 'pvcz_t_rack',
       'pvcz_t_roof', 'pvcz_humidity', 'pvcz_wind', 'tracking', 'type',
       'azimuth', 'tilt', 'first_timestamp', 'last_timestamp', 'years',
       'number_records', 'dataset_size_mb', 'available_sensor_channels',
       'qa_status', 'qa_issue', 'first_year', 'is_prize_data',
       'is_lake_parquet_data', 'is_lake_csv_data', 'has_irradiance_data',
       'has_ambient_temperature_data', 'has_temperature_data',
       'has_power_data', 'has_current_data', 'has_voltage_data', 'has_ac_data',
       'has_dc_data', 'module_type', 'simplified_type', 'system_source',
       'num_days_actual_records'],
      dtype='str')

In [3]:
metrics_dir = Path("../../../data/raw/parquet-metrics/")
metrics_pq = pq.ParquetDataset(metrics_dir)
metrics_df = metrics_pq.read().to_pandas()
metrics_id_set = set(metrics_df.system_id)

In [4]:
ac_power_metrics = metrics_search_for_two_fragments_df(metrics_df, 'ac', 'pow', 'and')
ac_power_systems = set(ac_power_metrics['system_id'])
two_years_days = 2 * 365
enough_data_systems = systems_cleaned[systems_cleaned['num_days_actual_records'] >= two_years_days]
enough_data_ids = set(enough_data_systems.system_id)
enough_data_parquet_power_systems = enough_data_ids.intersection(ac_power_systems)
enough_data_parquet_power_list = list(enough_data_parquet_power_systems)
enough_data_parquet_power_list.sort()

## Early practice

In [106]:
system_id = 1200
test_folder_str = f'../../../../data_ds_project/testing_yearly_parquet/{system_id}'
test_folder = Path(test_folder_str)

In [107]:
test_pq = pq.ParquetDataset(test_folder)
test_df = test_pq.read().to_pandas()

In [58]:
test_df

,time,ac_power_kW_inverter,ac_power_kW_meter,year
0,2010-10-03 12:15:12,7.052258,NaN,2010
1,2010-10-03 12:30:12,9.621044,NaN,2010
2,2010-10-03 12:45:12,8.489045,NaN,2010
3,2010-10-03 13:00:13,9.352703,NaN,2010
4,2010-10-03 13:15:13,16.757570,NaN,2010
...,...,...,...,...
59796,2020-07-26 16:40:00,NaN,29.38,2020
59797,2020-07-26 16:45:00,NaN,28.30,2020
59798,2020-07-26 16:50:00,NaN,27.90,2020
59799,2020-07-26 16:55:00,NaN,27.06,2020


### quick check -- are there all columns with per-year access?

In [61]:
just_inverter_one_year = pq.ParquetDataset(
    test_folder,
    filters=[
        ('year', '==', 2011)
    ]
)
just_inverter_one_year_df = just_inverter_one_year.read().to_pandas()

In [62]:
just_inverter_one_year_df.head()

,time,ac_power_kW_inverter,ac_power_kW_meter,year
0,2011-01-01 07:30:06,0.000574,NaN,2011
1,2011-01-01 07:45:06,0.159709,NaN,2011
2,2011-01-01 08:00:06,0.442825,NaN,2011
3,2011-01-01 08:15:06,1.773831,NaN,2011
4,2011-01-01 08:30:06,3.322117,NaN,2011


In [63]:
just_bad_year = pq.ParquetDataset(
    test_folder,
    filters=[
        ('year', '==', 1994)
    ]
)
just_bad_df = just_bad_year.read().to_pandas()
just_bad_df

,time,ac_power_kW_inverter,ac_power_kW_meter,year


Phew, all columns included in all years, even empty-data ones!

In [59]:
test_df['year'].value_counts()

year
2015    174783
2014    174639
2016    110139
2017    105121
2019    105100
2018    105088
2020     59801
2013     50663
2012     34009
2011     19392
2010      3763
Name: count, dtype: int64

In [8]:
test_df.columns[test_df.columns.str.contains('inv')]

Index(['ac_power_kW_inverter'], dtype='str', name='')

In [5]:
def standardize(df: pd.DataFrame,
                null_or_zero: str,
                system_id: int,
                systems_cleaned: pd.DataFrame,
                drop_na: bool):
    '''Standardize input dataframes for univariate analysis.
    
    Parameters
    -----------
    df: pandas.DataFrame
        The dataframe output from ac_power_parquet_distiller_yearly.py
        or ac_power_parquet_distiller.py
    null_or_zero: str, "null", "zero", or "raw"
        Determine what to do with data less than 1 percent
        of maximum value
        If "null", replace small values by numpy.nan
        If "zero", replace with zero.
        If "raw", skip this cleaning step
        If anything else, throw a ValueError.
    system_id: int
        The system_id of the system.
    systems_cleaned: pd.DataFrame
        The metadata dataframe.
    drop_na: bool
        Determine whether or not to drop NA terms

    Returns
    ---------
    A list of one or two DataFrames.
    If both inverter and meter in input, get a list of two DataFrames
    If only one power input, get a list of one DataFrame
    Otherwise, bad input, ValueError
    '''
    # copy input for future use.
    df = df.copy(deep=True)
    # grab column names
    col_names = df.columns
    pow_col_names = col_names[col_names.str.contains('ac_power_kW')]
    # grab official max. value.
    relevant_rows_systems = systems_cleaned[systems_cleaned['system_id'] == system_id]
    first_index = relevant_rows_systems.index[0]
    max_dc_capacity = relevant_rows_systems.loc[first_index, 'dc_capacity_kW']
    for pow_col_name in pow_col_names:
        local_max = df[pow_col_name].max()
        smaller_max = max(min(local_max, max_dc_capacity), 0)
        lower_bound = 0.01 * smaller_max
        if null_or_zero == 'null':
            df[pow_col_name] = df.apply(
                lambda row: row[pow_col_name]
                if (row[pow_col_name] is not pd.NA
                    and row[pow_col_name] is not np.nan
                    and row[pow_col_name] >= lower_bound)
                else np.nan,  # not pandas.NA with float64 dtype
                axis = 1
            )
        elif null_or_zero == 'zero':
            df[pow_col_name] = df.apply(
                lambda row: row[pow_col_name]
                if (row[pow_col_name] is not pd.NA
                    and row[pow_col_name] is not np.nan
                    and row[pow_col_name] >= lower_bound)
                else 0,
                axis = 1
            )
        elif null_or_zero == 'raw':
            pass
        else:
            raise ValueError(f'null_or_zero, input {null_or_zero}, is none of'
                             + '"null", "zero", or "raw".')
    # if both inverter and meter, make two columns
    if (
        any(pow_col_names.str.contains('inv'))
        and any(pow_col_names.str.contains('met'))
    ):
        # need two frames
        inv_pow_names = pow_col_names[pow_col_names.str.contains('inv')]
        if len(inv_pow_names) == 1:
            inv_pow_name = inv_pow_names[0]
        else:
            raise ValueError('Bad input dataframe -- multiple inverter columns!')
        met_pow_names = pow_col_names[pow_col_names.str.contains('met')]
        if len(met_pow_names) == 1:
            met_pow_name = met_pow_names[0]
        else:
            raise ValueError('Bad input dataframe -- multiple meter columns!')
        inv_data = df[['time', inv_pow_name]].copy(deep=True)
        met_data = df[['time', met_pow_name]].copy(deep=True)
        inv_data = inv_data.drop_duplicates()
        met_data = met_data.drop_duplicates()
        if drop_na:
            inv_data = inv_data.dropna()
            met_data = met_data.dropna()
        # rename columns
        renamer = {
            inv_pow_name: 'power',
            met_pow_name: 'power'
        }
        inv_data = inv_data.rename(columns=renamer)
        met_data = met_data.rename(columns=renamer)
        return [inv_data, met_data]
    elif len(pow_col_names) == 1:
        pow_col_name = pow_col_names[0]
        my_data = df[['time', pow_col_name]].copy(deep=True)
        renamer = {
            pow_col_name: 'power',
        }
        my_data = my_data.rename(columns=renamer)
        my_data = my_data.drop_duplicates()
        if drop_na:
            my_data = my_data.dropna()
        return [my_data,]
    else:
        raise ValueError('Not expected input!')

Good, but still requires too much data for year-by-year.  Perhaps there is a better way to do year-by-year,
since below experimentation ensures that the columns are the same no matter if data is missing or empty.

In [6]:
def standardize_single_year(
    input_dir,
    null_or_zero: str,
    system_id: int,
    systems_cleaned: pd.DataFrame,
    year: int,
    drop_na: bool
):
    '''Standardize input dataframes for univariate analysis.
    
    Parameters
    -----------
    input_dir: str or pathlib.Path or os.Path
        String or Path object representing the path to the data.
        from ac_power_parquet_distiller_yearly.py
    null_or_zero: str, "null", "zero", or "raw"
        Determine what to do with data less than 1 percent
        of maximum value
        If "null", replace small values by numpy.nan
        If "zero", replace with zero.
        If "raw", skip this cleaning step
        If anything else, throw a ValueError.
    system_id: int
        The system_id of the system.
    systems_cleaned: pd.DataFrame
        The metadata dataframe.
    drop_na: bool
        Determine whether or not to drop NA terms

    Returns
    ---------
    A list of one or two DataFrames.
    If both inverter and meter in input, get a list of two DataFrames
    If only one power input, get a list of one DataFrame
    Otherwise, bad input, ValueError
    '''
    if type(input_dir) == str:
        input_dir = Path(input_dir)
    # grab the data
    current_year_pq = pq.ParquetDataset(
        input_dir,
        filters=[('year', '==', year)]
    )
    current_year_df = current_year_pq.read().to_pandas()
    # grab column names
    col_names = current_year_df.columns
    pow_col_names = col_names[col_names.str.contains('ac_power_kW')]
    # grab official max. value.
    relevant_rows_systems = systems_cleaned[systems_cleaned['system_id'] == system_id]
    first_index = relevant_rows_systems.index[0]
    max_dc_capacity = relevant_rows_systems.at[first_index, 'dc_capacity_kW']
    for pow_col_name in pow_col_names:
        local_max = current_year_df[pow_col_name].max()
        smaller_max = max(min(local_max, max_dc_capacity), 0)
        lower_bound = 0.01 * smaller_max
        if null_or_zero == 'null':
            current_year_df[pow_col_name] = current_year_df.apply(
                lambda row: row[pow_col_name]
                if (row[pow_col_name] is not pd.NA
                    and row[pow_col_name] is not np.nan
                    and row[pow_col_name] >= lower_bound)
                else np.nan,  # not pandas.NA with float64 dtype
                axis = 1
            )
        elif null_or_zero == 'zero':
            current_year_df[pow_col_name] = current_year_df.apply(
                lambda row: row[pow_col_name]
                if (row[pow_col_name] is not pd.NA
                    and row[pow_col_name] is not np.nan
                    and row[pow_col_name] >= lower_bound)
                else 0,
                axis = 1
            )
        elif null_or_zero == 'raw':
            pass
        else:
            raise ValueError(f'null_or_zero, input {null_or_zero}, is none of'
                             + '"null", "zero", or "raw".')
    # if both inverter and meter, make two columns
    if (
        any(pow_col_names.str.contains('inv'))
        and any(pow_col_names.str.contains('met'))
    ):
        # need two frames
        inv_pow_names = pow_col_names[pow_col_names.str.contains('inv')]
        if len(inv_pow_names) == 1:
            inv_pow_name = inv_pow_names[0]
        else:
            raise ValueError('Bad input dataframe -- multiple inverter columns!')
        met_pow_names = pow_col_names[pow_col_names.str.contains('met')]
        if len(met_pow_names) == 1:
            met_pow_name = met_pow_names[0]
        else:
            raise ValueError('Bad input dataframe -- multiple meter columns!')
        inv_data = current_year_df[['time', inv_pow_name]].copy(deep=True)
        met_data = current_year_df[['time', met_pow_name]].copy(deep=True)
        inv_data = inv_data.drop_duplicates()
        met_data = met_data.drop_duplicates()
        if drop_na:
            inv_data = inv_data.dropna()
            met_data = met_data.dropna()
        # rename columns
        renamer = {
            inv_pow_name: 'power',
            met_pow_name: 'power'
        }
        inv_data = inv_data.rename(columns=renamer)
        met_data = met_data.rename(columns=renamer)
        return [inv_data, met_data]
    elif len(pow_col_names) == 1:
        pow_col_name = pow_col_names[0]
        my_data = current_year_df[['time', pow_col_name]].copy(deep=True)
        renamer = {
            pow_col_name: 'power',
        }
        my_data = my_data.rename(columns=renamer)
        my_data = my_data.drop_duplicates()
        if drop_na:
            my_data = my_data.dropna()
        return [my_data,]
    else:
        raise ValueError('Not expected input!')

In [103]:
sample_33, = standardize_single_year(
    '../../../../data_ds_project/testing_yearly_parquet/33',
    'null',
    33,
    systems_cleaned,
    2013,
    True
)

In [104]:
sample_33

,time,power
444,2013-01-01 07:24:00,0.031410
445,2013-01-01 07:25:00,0.034227
446,2013-01-01 07:26:00,0.034759
447,2013-01-01 07:27:00,0.033406
448,2013-01-01 07:28:00,0.030042
...,...,...
508379,2013-12-31 16:36:00,0.047461
508380,2013-12-31 16:37:00,0.042340
508381,2013-12-31 16:38:00,0.035821
508382,2013-12-31 16:39:00,0.030041


In [108]:
inv_out, met_out = standardize(test_df,
    'null',
    system_id,
    systems_cleaned,
    True
)

In [70]:
type('abcdefg')

str

In [109]:
inv_out.describe()

,time,power
count,81103,81103.000000
mean,2012-08-13 19:14:24.518612480,16.438740
min,2010-10-03 12:15:12,0.486743
25%,2012-01-03 16:18:01,4.132064
50%,2012-07-15 06:35:31,12.693110
75%,2013-06-27 11:03:03,28.342121
max,2013-12-01 16:15:32,48.662719
std,NaN,13.346824


In [35]:
met_out.describe()

,time,power
count,146281,146281.000000
mean,2017-09-09 23:25:20.977242112,15.688420
min,2013-11-30 07:24:00,0.500000
25%,2014-06-04 08:33:00,3.560000
50%,2019-01-28 09:45:00,10.860000
75%,2019-10-17 12:05:00,27.340000
max,2020-07-26 17:00:00,48.160000
std,NaN,13.713706


In [36]:
inv_out['time'].diff().value_counts()

time
0 days 00:05:00    55545
0 days 00:15:00    13296
0 days 00:05:01     3944
0 days 00:15:01     2654
0 days 00:04:59     2001
                   ...  
0 days 14:50:02        1
0 days 15:04:58        1
0 days 15:40:00        1
0 days 15:14:55        1
0 days 17:09:56        1
Name: count, Length: 679, dtype: int64

In [80]:
inv_out['time'].diff().head()

0               NaT
1   0 days 00:15:00
2   0 days 00:15:00
3   0 days 00:15:01
4   0 days 00:15:00
Name: time, dtype: timedelta64[ns]

In [81]:
inv_out['time'].diff().tail()

36087   0 days 00:05:00
36090   0 days 00:05:00
36092   0 days 00:05:00
36095   0 days 00:05:00
36098   0 days 00:05:00
Name: time, dtype: timedelta64[ns]

In [39]:
met_out['time'].diff().value_counts()

time
0 days 00:05:00    91603
0 days 00:03:00    52386
0 days 00:04:36      328
0 days 00:05:24      325
0 days 00:10:00      118
                   ...  
0 days 00:05:22        1
0 days 00:04:38        1
0 days 00:05:45        1
0 days 00:04:15        1
0 days 02:10:00        1
Name: count, Length: 314, dtype: int64

In [77]:
met_out['time'].diff().head()

35213               NaT
35215   0 days 00:03:00
35216   0 days 00:03:00
35218   0 days 00:03:00
35220   0 days 00:03:00
Name: time, dtype: timedelta64[ns]

In [78]:
met_out['time'].diff().tail()

59796   0 days 00:05:00
59797   0 days 00:05:00
59798   0 days 00:05:00
59799   0 days 00:05:00
59800   0 days 00:05:00
Name: time, dtype: timedelta64[ns]

In [51]:
all_times_diff = test_df['time'].diff()
all_small_diff = all_times_diff[all_times_diff < timedelta(seconds=3600)]
all_small_diff.value_counts()

time
0 days 00:05:00    534056
0 days 00:03:00    376456
0 days 00:15:00     15515
0 days 00:05:01      4436
0 days 00:15:01      3083
                    ...  
0 days 00:46:44         1
0 days 00:03:16         1
0 days 00:05:51         1
0 days 00:04:09         1
0 days 00:05:21         1
Name: count, Length: 229, dtype: int64

In [50]:
test_df['time'].diff().value_counts()

time
0 days 00:05:00    534056
0 days 00:03:00    376456
0 days 00:15:00     15515
0 days 00:05:01      4436
0 days 00:15:01      3083
                    ...  
0 days 00:46:44         1
0 days 00:03:16         1
0 days 00:05:51         1
0 days 00:04:09         1
0 days 00:05:21         1
Name: count, Length: 752, dtype: int64

### Let me try a different system.  Could just be the off-and-on here

In [52]:
system_id = 4902
test_folder_str = f'../../../../data_ds_project/testing_yearly_parquet/{system_id}'
test_folder = Path(test_folder_str)


In [53]:
nist_pq = pq.ParquetDataset(test_folder)
nist_df = nist_pq.read().to_pandas()

In [54]:
nist_df_times = nist_df['time'].diff()
nist_df_times.value_counts()

time
0 days 00:01:00    1871917
0 days 00:03:00       3438
0 days 00:02:00       2524
0 days 00:04:00        912
0 days 00:05:00        440
0 days 00:06:00        254
0 days 00:07:00        177
0 days 00:08:00        122
0 days 00:09:00         73
0 days 00:10:00         57
0 days 00:11:00         40
0 days 00:12:00         33
0 days 00:13:00         24
0 days 00:14:00         14
0 days 00:15:00         13
0 days 00:17:00         12
0 days 00:18:00          8
0 days 00:16:00          5
0 days 00:21:00          5
0 days 00:22:00          4
0 days 00:20:00          4
0 days 00:19:00          3
0 days 00:23:00          2
0 days 12:36:00          1
0 days 17:13:00          1
2 days 03:55:00          1
0 days 10:36:00          1
0 days 00:41:00          1
0 days 00:24:00          1
0 days 00:28:00          1
0 days 00:31:00          1
1 days 00:01:00          1
Name: count, dtype: int64

As expected, minute-by-minute updates.  Much nicer!

In [55]:
nist_df_late = nist_df[nist_df['time'] >= datetime(2017, 1, 1)]
nist_df_late_times = nist_df_late['time'].diff()
nist_df_late_times.value_counts() 

time
0 days 00:01:00    627696
0 days 00:02:00       322
0 days 00:03:00       164
0 days 00:04:00       140
0 days 00:05:00        81
0 days 00:07:00        27
0 days 00:06:00        22
0 days 00:08:00        20
0 days 00:09:00        11
0 days 00:10:00         8
0 days 00:13:00         6
0 days 00:11:00         5
0 days 00:12:00         4
0 days 00:17:00         2
0 days 00:14:00         1
0 days 00:18:00         1
0 days 00:15:00         1
Name: count, dtype: int64

### Another system

In [56]:
system_id = 4
test_folder_str = f'../../../../data_ds_project/testing_yearly_parquet/{system_id}'
test_folder = Path(test_folder_str)
test_pq = pq.ParquetDataset(test_folder)
test_df = test_pq.read().to_pandas()

In [57]:
test_df.head()

,time,ac_power_kW,year
0,2007-08-26 00:00:00,-0.000006,2007
1,2007-08-26 00:15:00,-0.000029,2007
2,2007-08-26 00:30:00,-0.000029,2007
3,2007-08-26 00:45:00,-0.000020,2007
4,2007-08-26 01:00:00,-0.000021,2007


In [58]:
good_data, = standardize(
    test_df,
    'null',
    system_id,
    systems_cleaned,
    True
)

In [59]:
good_data['time'].diff().value_counts()

time
0 days 00:01:00    2522600
0 days 00:15:00      24565
0 days 00:07:00        858
0 days 00:06:00        502
0 days 00:02:00        481
                    ...   
1 days 15:54:00          1
1 days 17:22:00          1
0 days 16:24:00          1
1 days 16:21:00          1
0 days 16:33:00          1
Name: count, Length: 750, dtype: int64

In [65]:
time_diffs = good_data['time'].diff()
short_diffs = time_diffs[time_diffs <= timedelta(seconds=3600)]
common_length = short_diffs.value_counts().index[0]

In [76]:
short_diffs.value_counts()

time
0 days 00:01:00    2522600
0 days 00:15:00      24565
0 days 00:07:00        858
0 days 00:06:00        502
0 days 00:02:00        481
0 days 00:08:00        322
0 days 00:03:00        300
0 days 00:04:00        192
0 days 00:05:00        143
0 days 00:30:00         80
0 days 00:09:00         73
0 days 00:10:00         67
0 days 00:14:00         49
0 days 00:11:00         49
0 days 00:13:00         48
0 days 00:12:00         44
0 days 00:21:00         42
0 days 00:16:00         37
0 days 00:20:00         32
0 days 00:17:00         30
0 days 00:45:00         27
0 days 00:18:00         27
0 days 00:22:00         25
0 days 00:27:00         25
0 days 00:19:00         23
0 days 00:24:00         22
0 days 00:23:00         22
0 days 00:26:00         21
0 days 00:25:00         21
0 days 00:28:00         21
0 days 00:29:00         19
0 days 00:32:00         17
0 days 00:31:00         16
0 days 00:33:00         14
0 days 00:47:00         13
0 days 00:42:00         12
0 days 00:53:00        

In [73]:
time_diffs

57                    NaT
58        0 days 00:15:00
59        0 days 00:15:00
60        0 days 00:15:00
61        0 days 00:15:00
                ...      
6342280   0 days 00:01:00
6342281   0 days 00:01:00
6342282   0 days 00:01:00
6342283   0 days 00:01:00
6342284   0 days 00:01:00
Name: time, Length: 2555751, dtype: timedelta64[ns]

In [84]:
time_diffs.name = 'Delta_t'

In [85]:
time_collect = pd.merge(left = test_df['time'], right = time_diffs, left_index=True, right_index=True)

In [86]:
time_collect

,time,Delta_t
57,2007-08-26 14:15:00,NaT
58,2007-08-26 14:30:00,0 days 00:15:00
59,2007-08-26 14:45:00,0 days 00:15:00
60,2007-08-26 15:00:00,0 days 00:15:00
61,2007-08-26 15:15:00,0 days 00:15:00
...,...,...
6342280,2023-02-28 17:13:00,0 days 00:01:00
6342281,2023-02-28 17:14:00,0 days 00:01:00
6342282,2023-02-28 17:15:00,0 days 00:01:00
6342283,2023-02-28 17:16:00,0 days 00:01:00


In [66]:
common_length

Timedelta('0 days 00:01:00')

In [72]:
short_diffs.head()

58   0 days 00:15:00
59   0 days 00:15:00
60   0 days 00:15:00
61   0 days 00:15:00
62   0 days 00:15:00
Name: time, dtype: timedelta64[ns]

In [67]:
common_length_ends = short_diffs[(short_diffs <= common_length + timedelta(seconds=1))
                                 & (short_diffs >= common_length - timedelta(seconds=1))]

In [71]:
common_length_ends.iloc[-5:-1]

6342280   0 days 00:01:00
6342281   0 days 00:01:00
6342282   0 days 00:01:00
6342283   0 days 00:01:00
Name: time, dtype: timedelta64[ns]

### Again!

In [35]:
system_id = 1300
test_folder_str = f'../../../../data_ds_project/testing_yearly_parquet/{system_id}'
test_folder = Path(test_folder_str)
test_pq = pq.ParquetDataset(test_folder)
test_df = test_pq.read().to_pandas()

In [36]:
test_df.head()

,time,ac_power_kW,year
0,2013-02-14 07:15:00,0.00000,2013
1,2013-02-14 07:20:00,0.00098,2013
2,2013-02-14 07:25:00,0.00915,2013
3,2013-02-14 07:30:00,0.02678,2013
4,2013-02-14 07:35:00,0.07955,2013


In [37]:
loc_data, = standardize(
    test_df,
    null_or_zero='null',
    system_id=1300,
    systems_cleaned=systems_cleaned,
    drop_na=True
)

In [38]:
loc_data.head()

,time,power
3,2013-02-14 07:30:00,0.02678
4,2013-02-14 07:35:00,0.07955
5,2013-02-14 07:40:00,0.12767
6,2013-02-14 07:45:00,0.12524
7,2013-02-14 07:50:00,0.06745


In [40]:
loc_data['time'].dt.year

3         2013
4         2013
5         2013
6         2013
7         2013
          ... 
203234    2018
203235    2018
203236    2018
203237    2018
203238    2018
Name: time, Length: 99276, dtype: int32

In [96]:
loc_data.loc[:, 'Delta_t'] = loc_data['time'].diff()

In [98]:
loc_data['Delta_t'].value_counts()

Delta_t
0 days 00:05:00     98344
0 days 00:10:00        81
0 days 00:15:00        39
0 days 11:15:00        39
0 days 11:40:00        29
                    ...  
0 days 02:05:00         1
28 days 12:05:00        1
0 days 02:40:00         1
0 days 02:20:00         1
0 days 22:30:00         1
Name: count, Length: 96, dtype: int64

In [69]:
def same_start_and_end_diff(
    system_id: int,
    systems_cleaned: pd.DataFrame
):
    # get and standardize data
    test_folder_str = f'../../../../data_ds_project/testing_yearly_parquet/{system_id}/'
    test_folder = Path(test_folder_str)
    test_pq = pq.ParquetDataset(test_folder)
    test_df = test_pq.read().to_pandas()
    my_data_list = standardize(
        test_df,
        null_or_zero='null',
        system_id=system_id,
        systems_cleaned=systems_cleaned,
        drop_na=True
    )
    if my_data_list is None or (len(my_data_list) == 0):
        raise ValueError('No data for this system!')
    out_list = []
    for df in my_data_list:
        early_data = df.iloc[0:1000]
        early_data_time_diffs = early_data['time'].diff()
        common_early_time = early_data_time_diffs.value_counts().index[0]
        late_data = df.iloc[-1000:]
        late_data_time_diffs = late_data['time'].diff()
        common_late_time = late_data_time_diffs.value_counts().index[0]
        out_list.append(
            ((common_early_time - common_late_time) < timedelta(seconds=5))
            and ((common_late_time - common_early_time) < timedelta(seconds=5))
        )
    return out_list

In [50]:
data_dict = dict()
for system_id in tqdm(enough_data_parquet_power_list):
    test_results = same_start_and_end_diff(
        system_id=system_id, systems_cleaned=systems_cleaned
    )
    if len(test_results) == 2:
        data_dict[(system_id, 'inverter')] = test_results[0]
        data_dict[(system_id, 'meter')] = test_results[1]
    elif len(test_results) == 1:
        data_dict[(system_id, 'unknown')] = test_results[0]
    else:
        raise RuntimeError('Bad results from same_start_and_end_diff(*args)!')
out_df = pd.Series(data = data_dict, name='start_and_end_agreement')

100%|██████████| 82/82 [25:24<00:00, 18.59s/it]  


In [107]:
out_df.head()

4   unknown    False
10  unknown    False
33  unknown     True
34  unknown     True
35  unknown     True
Name: start_and_end_agreement, dtype: bool

In [109]:
out_df.index

MultiIndex([(   4,  'unknown'),
            (  10,  'unknown'),
            (  33,  'unknown'),
            (  34,  'unknown'),
            (  35,  'unknown'),
            (  36,  'unknown'),
            (  50,  'unknown'),
            (  51,  'unknown'),
            (1199,  'unknown'),
            (1200, 'inverter'),
            (1200,    'meter'),
            (1201,  'unknown'),
            (1202, 'inverter'),
            (1202,    'meter'),
            (1203, 'inverter'),
            (1203,    'meter'),
            (1204,  'unknown'),
            (1208, 'inverter'),
            (1208,    'meter'),
            (1214,  'unknown'),
            (1217,  'unknown'),
            (1219,  'unknown'),
            (1221,  'unknown'),
            (1223,  'unknown'),
            (1225,  'unknown'),
            (1231,  'unknown'),
            (1239,  'unknown'),
            (1244,  'unknown'),
            (1246,  'unknown'),
            (1248,  'unknown'),
            (1249,  'unknown'),
        

In [51]:
out_df.index.names = ['system_id', 'source_type']

In [111]:
out_df

system_id  source_type
4          unknown        False
10         unknown        False
33         unknown         True
34         unknown         True
35         unknown         True
                          ...  
4901       meter           True
4902       inverter        True
           meter           True
4903       inverter        True
           meter           True
Name: start_and_end_agreement, Length: 91, dtype: bool

In [52]:
out_df.columns = ['consistent_start_and_end']

In [115]:
out_df.head()

system_id  source_type
4          unknown        False
10         unknown        False
33         unknown         True
34         unknown         True
35         unknown         True
Name: start_and_end_agreement, dtype: bool

In [116]:
out_df.value_counts()

start_and_end_agreement
True     76
False    15
Name: count, dtype: int64

In [54]:
out_df[~out_df]

system_id  source_type
4          unknown        False
10         unknown        False
50         unknown        False
51         unknown        False
1199       unknown        False
1200       inverter       False
           meter          False
1201       unknown        False
1202       meter          False
1203       inverter       False
           meter          False
1308       unknown        False
1418       unknown        False
1419       unknown        False
1420       unknown        False
Name: start_and_end_agreement, dtype: bool

In [70]:
def year_by_year_common_timesteps(
    system_id: int,
    systems_cleaned: pd.DataFrame
):
    # initialize data frame
    # know all data is in 1994 - 2023 range
    num_years = 2023 - 1994 + 1
    output_df = pd.DataFrame(
        data = np.full((num_years, 3), fill_value=pd.NA),
        index = range(1994, 2024),
        columns = ['inverter', 'meter', 'unknown'],
        dtype='object'
    )
    # for ease of use, since some readings come and go,
    # grab all the data at once, still, to avoid messy constructions.
    test_folder_str = f'../../../../data_ds_project/testing_yearly_parquet/{system_id}/'
    test_folder = Path(test_folder_str)
    test_pq = pq.ParquetDataset(test_folder)
    test_df = test_pq.read().to_pandas()
    my_data_list = standardize(
        test_df,
        null_or_zero='null',
        system_id=system_id,
        systems_cleaned=systems_cleaned,
        drop_na=True
    )
    if my_data_list is None or (len(my_data_list) == 0):
        raise ValueError('No data for this system!')
    elif len(my_data_list) == 2: # inverter/meter split
        inv_data = my_data_list[0]
        met_data = my_data_list[1]
        for year in range(1994, 2024):
            inv_data_year = inv_data[inv_data['time'].dt.year == np.int32(year)]
            if inv_data_year.shape[0] > 5:
                inv_data_year_time_diffs = inv_data_year['time'].diff()
                inv_data_year_common_diff = inv_data_year_time_diffs.value_counts().index[0]
                output_df.at[year, 'inverter'] = inv_data_year_common_diff
            met_data_year = met_data[met_data['time'].dt.year == np.int32(year)]
            if met_data_year.shape[0] > 5:
                met_data_year_time_diffs = met_data_year['time'].diff()
                met_data_year_common_diff = met_data_year_time_diffs.value_counts().index[0]
                output_df.at[year, 'meter'] = met_data_year_common_diff
    elif len(my_data_list) == 1:
        unk_data = my_data_list[0]
        for year in range(1994, 2024):
            unk_data_year = unk_data[unk_data['time'].dt.year == np.int32(year)]
            if unk_data_year.shape[0] > 5:
                data_year_time_diffs = unk_data_year['time'].diff()
                data_year_common_diff = data_year_time_diffs.value_counts().index[0]
                output_df.at[year, 'unknown'] = data_year_common_diff
    else:
        raise ValueError('Bad input dataframe given.')
    
    # drop irrelevant columns
    output_df = output_df.dropna(axis=1, how='all')
    # drop irrelevant rows
    output_df = output_df.dropna(axis=0, how='all')
    # don't bother with empty dataframes
    if output_df.shape[0] == 0:
        return None
    else:
        return output_df

In [49]:
for system_id in enough_data_parquet_power_list:
    print(system_id)
    results_df = year_by_year_common_timesteps(
        system_id=system_id, systems_cleaned=systems_cleaned
    )
    print(results_df)
    print('')

4
              unknown
2007  0 days 00:15:00
2008  0 days 00:15:00
2009  0 days 00:15:00
2010  0 days 00:01:00
2011  0 days 00:01:00
2012  0 days 00:01:00
2013  0 days 00:01:00
2014  0 days 00:01:00
2015  0 days 00:01:00
2016  0 days 00:01:00
2017  0 days 00:01:00
2018  0 days 00:01:00
2019  0 days 00:01:00
2020  0 days 00:01:00
2021  0 days 00:01:00
2022  0 days 00:01:00
2023  0 days 00:01:00

10
              unknown
2006  0 days 00:15:00
2007  0 days 00:15:00
2008  0 days 00:15:00
2009  0 days 00:15:00
2010  0 days 00:01:00
2011  0 days 00:01:00
2012  0 days 00:01:00
2013  0 days 00:01:00
2014  0 days 00:01:00
2015  0 days 00:01:00
2016  0 days 00:01:00
2017  0 days 00:01:00
2018  0 days 00:01:00
2019  0 days 00:01:00
2020  0 days 00:01:00
2021  0 days 00:01:00
2022  0 days 00:01:00
2023  0 days 00:01:00

33
              unknown
2010  0 days 00:01:00
2011  0 days 00:01:00
2012  0 days 00:01:00
2013  0 days 00:01:00
2014  0 days 00:01:00
2015  0 days 00:01:00
2016  0 days 00:01:00


Results on transitions:

System 4: 2009 - 2010 somewhere

System 10: 2009 - 2010 transition

System 50: 2009 - 2011 somewhere (NO 2010 data)

System 51: 15-minute increments before 2009; 1-min. 1 increments common in 2011; 15-min. increments common in 2012-13,
1-minute increments after that

System 1199: Starts 15 min. increment in 2010, by the next year switches to 5 min.

System 1200, Inverter: 2011 - 2012 somewhere

System 1200, Meter: 3 min. before 2014, 5 min. after 2018!  (Status of data inbetween ???)

System 1201, 3 min. pre-2014, 5 min. post-2018 -- again, weird cuts in the middle!

System 1202, *Inverter* (not caught before): initial segment of 15-min. increment 1st year,
before 6 minute increment while data lasts.

System 1202, *Meter*: initial segment 3 min. before 2014, final segment 5 min. increment after 2018.  Inbetween we are not reading correctly!

System 1203, Inverter and Meter: 3 min. before 2015, 5 min. after 2016 (maybe).

System 1308 is very strange -- looks like 15 min. standard every year.  False flag with 1st 1000 and last 1000 points, or just bad data?

System 1418: 2016 - 2017 changeover

System 1419: 2017 - 2018 changeover

System 1420: 2016 - 2017 changeover

## Try monthly, direct version [less cleaning, just power]

In [8]:
non_standard_systems = [4, 10, 50, 51, 1199, 1200, 1201, 1202, 1203, 1418, 1419, 1420]

In [9]:
def year_by_year_direct(
    system_id: int,
    systems_cleaned: pd.DataFrame
):
    # initialize data frame
    # know all data is in 1994 - 2023 range
    num_years = 2023 - 1994 + 1
    output_df = pd.DataFrame(
        data = np.full((num_years, 3), fill_value=pd.NA),
        index = range(1994, 2024),
        columns = ['inverter', 'meter', 'unknown'],
        dtype='object'
    )
    # Try year-by-year
    test_folder_str = f'../../../../data_ds_project/testing_yearly_parquet/{system_id}/'
    test_folder = Path(test_folder_str)
    # columns invariant among years accessed, so can just grab a year with no data to grab column names
    empty_year_pq = pq.ParquetDataset(
        test_folder,
        filters=[('year', '==', 1990)]
    )
    empty_year_df = empty_year_pq.read().to_pandas()

    # grab column names
    col_names = empty_year_df.columns
    pow_col_names = col_names[col_names.str.contains('ac_power_kW')]
    for year in range(1994, 2024):
        current_year_pq = pq.ParquetDataset(
            test_folder,
            filters=[
                ('year', '==', year)
            ]            
        )
        current_year_df = current_year_pq.read().to_pandas()
        for pow_col_name in pow_col_names:
            pow_selection = current_year_df[['time', pow_col_name]]
            # drop if no data in the column
            pow_selection = pow_selection.dropna(axis=1, how='all')
            if (pow_selection.shape[1] > 1) and (pow_selection.shape[0] > 10):
                pow_selection_time_diffs = pow_selection['time'].diff()
                pow_selection_commonest_time = pow_selection_time_diffs.value_counts().index[0]
                if 'inv' in pow_col_name:
                    output_df.at[year, 'inverter'] = pow_selection_commonest_time
                elif 'met' in pow_col_name:
                    output_df.at[year, 'meter'] = pow_selection_commonest_time
                else:
                    output_df.at[year, 'unknown'] = pow_selection_commonest_time
    # drop irrelevant columns
    output_df = output_df.dropna(axis=1, how='all')
    # drop irrelevant rows
    output_df = output_df.dropna(axis=0, how='all')
    # don't bother with empty dataframes
    if output_df.shape[0] == 0:
        return None
    else:
        return output_df
    

In [79]:
for system_id in non_standard_systems:
    print(system_id)
    results_df = year_by_year_direct(
        system_id=system_id, systems_cleaned=systems_cleaned
    )
    print(results_df)
    print('')

4
              unknown
2007  0 days 00:15:00
2008  0 days 00:15:00
2009  0 days 00:15:00
2010  0 days 00:01:00
2011  0 days 00:01:00
2012  0 days 00:01:00
2013  0 days 00:01:00
2014  0 days 00:01:00
2015  0 days 00:01:00
2016  0 days 00:01:00
2017  0 days 00:01:00
2018  0 days 00:01:00
2019  0 days 00:01:00
2020  0 days 00:01:00
2021  0 days 00:01:00
2022  0 days 00:01:00
2023  0 days 00:01:00

10
              unknown
2000  0 days 00:01:00
2006  0 days 00:15:00
2007  0 days 00:15:00
2008  0 days 00:15:00
2009  0 days 00:15:00
2010  0 days 00:01:00
2011  0 days 00:01:00
2012  0 days 00:01:00
2013  0 days 00:01:00
2014  0 days 00:01:00
2015  0 days 00:01:00
2016  0 days 00:01:00
2017  0 days 00:01:00
2018  0 days 00:01:00
2019  0 days 00:01:00
2020  0 days 00:01:00
2021  0 days 00:01:00
2022  0 days 00:01:00
2023  0 days 00:01:00

50
              unknown
1994  0 days 00:15:00
1995  0 days 00:15:00
1996  0 days 00:15:00
1997  0 days 00:15:00
1998  0 days 00:15:00
1999  0 days 00:15:00


#### For each change, list by earlier year.

In [92]:
change_years = {
    system_id: [] for system_id in non_standard_systems
}
for system_id in non_standard_systems:
    results_df = year_by_year_direct(
        system_id=system_id, systems_cleaned=systems_cleaned
    )
    num_rows = results_df.shape[0]
    for j in range(len(results_df.index) - 1):
        this_year = results_df.index[j]
        next_year = results_df.index[j + 1]
        if any(results_df.loc[this_year] != results_df.loc[next_year]):
            if this_year not in change_years[system_id]:
                change_years[system_id].append(this_year)
            if next_year not in change_years[system_id]:
                change_years[system_id].append(next_year)


In [94]:
change_years

{4: [2009, 2010],
 10: [np.int64(2000), np.int64(2006), np.int64(2009), np.int64(2010)],
 50: [np.int64(2009), np.int64(2011)],
 51: [np.int64(2009),
  np.int64(2011),
  np.int64(2012),
  np.int64(2014),
  np.int64(2015)],
 1199: [2010, 2011],
 1200: [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020],
 1201: [2015, 2016],
 1202: [2010, 2011, 2014, 2015, 2016, 2017, 2018, 2019, 2020],
 1203: [2015, 2016, 2017, 2018],
 1418: [2016, 2017],
 1419: [2016, 2017],
 1420: [2016, 2017]}

In [10]:
def monthly_changes(
    system_id: int,
    systems_cleaned: pd.DataFrame,
    change_years: dict,
):
    # initialize data frame
    # know all data is in 1994 - 2023 range
    my_change_years = [int(year) for year in change_years[system_id]]
    num_months = 12 * len(my_change_years)
    output_df = pd.DataFrame(
        data = np.full((num_months, 3), fill_value=pd.NA),
        index = pd.MultiIndex.from_product(
            [my_change_years, range(1, 13)]
        ),
        columns = ['inverter', 'meter', 'unknown'],
        dtype='object'
    )
    output_df.index.names = ['year', 'month']
    # set target
    test_folder_str = f'../../../../data_ds_project/testing_yearly_parquet/{system_id}/'
    test_folder = Path(test_folder_str)
    # columns invariant among years accessed, so can just grab a year with no data to grab column names
    empty_year_pq = pq.ParquetDataset(
        test_folder,
        filters=[('year', '==', 1990)]
    )
    empty_year_df = empty_year_pq.read().to_pandas()

    # grab column names
    col_names = empty_year_df.columns
    pow_col_names = col_names[col_names.str.contains('ac_power_kW')]
    for year in my_change_years:
        current_year_pq = pq.ParquetDataset(
            test_folder,
            filters=[
                ('year', '==', year)
            ]            
        )
        current_year_df = current_year_pq.read().to_pandas()
        for pow_col_name in pow_col_names:
            pow_selection = current_year_df[['time', pow_col_name]]
            # drop if no data in the column
            pow_selection = pow_selection.dropna(axis=1, how='all')
            if (pow_selection.shape[1] > 1) and (pow_selection.shape[0] > 10):
                pow_selection.loc[:, 'month'] = pow_selection['time'].dt.month
                for month in sorted(pow_selection['month'].unique()):
                    month_selection = pow_selection[pow_selection['month'] == month]
                    clean_month = int(month)
                    if (month_selection.shape[0] > 10):
                        month_selection.loc[:, 'delta_t'] = month_selection['time'].diff()
                        month_selection_commonest_time = month_selection['delta_t'].value_counts().index[0]
                        if 'inv' in pow_col_name:
                            output_df.at[(year, clean_month), 'inverter'] = month_selection_commonest_time
                        elif 'met' in pow_col_name:
                            output_df.at[(year, clean_month), 'meter'] = month_selection_commonest_time
                        else:
                            output_df.at[(year, clean_month), 'unknown'] = month_selection_commonest_time
    # drop irrelevant columns
    output_df = output_df.dropna(axis=1, how='all')
    # drop irrelevant rows
    output_df = output_df.dropna(axis=0, how='all')
    # don't bother with empty dataframes
    if output_df.shape[0] == 0:
        return None
    else:
        return output_df

In [103]:
for system_id in non_standard_systems:
    print(system_id)
    results_df = monthly_changes(
        system_id=system_id, systems_cleaned=systems_cleaned,
        change_years=change_years
    )
    print(results_df)
    print('')

4
                    unknown
year month                 
2009 1      0 days 00:15:00
     2      0 days 00:15:00
     3      0 days 00:15:00
     4      0 days 00:15:00
     5      0 days 00:15:00
     6      0 days 00:15:00
2010 1      0 days 00:01:00
     2      0 days 00:01:00
     3      0 days 00:01:00
     4      0 days 00:01:00
     5      0 days 00:01:00
     6      0 days 00:01:00
     7      0 days 00:01:00
     10     0 days 00:01:00
     11     0 days 00:01:00
     12     0 days 00:01:00

10
                    unknown
year month                 
2000 5      0 days 00:01:00
2006 1      0 days 00:15:00
     2      0 days 00:15:00
     3      0 days 00:15:00
     4      0 days 00:15:00
     5      0 days 00:15:00
     6      0 days 00:15:00
     7      0 days 00:15:00
     8      0 days 00:15:00
     9      0 days 00:15:00
     10     0 days 00:15:00
     11     0 days 00:15:00
     12     0 days 00:15:00
2009 1      0 days 00:15:00
     2      0 days 00:15:00
     3      0 

Results:

System 4:  2009-06 and before -- 15 minutes

2010-01 and after -- 1 minute

System 10: Ignore 2000 short data (1 min.)

2006 -- 2009-04: 15 minutes

2010-01 and onwards: 1 minute

System 50: Before 2009-04: 15 min.

after 2011-04 -- 1 minute

System 51: Before 2009-04 : 15 minutes

2011/04 - 2011/10?: 1 minute

2011/11? - 2014/08: 15 minutes

2015-03 onwards -- 1 minute

System 1199: Before 2011/06?, 15-minute increment

After 2011/07?, 5-minute increment

In [104]:
results_1200 = monthly_changes(
        system_id=1200, systems_cleaned=systems_cleaned,
        change_years=change_years
    )

In [114]:
results_1200['inverter'].dropna().diff()

year  month
2010  10                    NaT
      11        0 days 00:00:00
      12        0 days 00:00:00
2011  1         0 days 00:00:00
      2         0 days 00:00:00
      3         0 days 00:00:00
      4         0 days 00:00:00
      5         0 days 00:00:00
      6         0 days 00:00:00
      7         0 days 00:00:00
      8         0 days 00:00:00
      9         0 days 00:00:00
      10        0 days 00:00:00
      11        0 days 00:00:00
      12      -1 days +23:50:00
2012  1         0 days 00:00:00
      2         0 days 00:00:00
      3         0 days 00:00:00
      4         0 days 00:00:00
      5         0 days 00:00:00
      6         0 days 00:00:00
      7         0 days 00:00:00
      8         0 days 00:00:00
      9         0 days 00:00:00
      10        0 days 00:00:00
      11        0 days 00:00:00
      12        0 days 00:00:00
2013  1         0 days 00:00:00
      2         0 days 00:00:00
      3         0 days 00:00:00
      4         0 days 00:00

In [117]:
meter_1200_diffs = results_1200['meter'].dropna().diff()
meter_1200_diffs[~(meter_1200_diffs == timedelta(days=0))]

year  month
2013  1                     NaT
      12      -1 days +23:58:00
2016  2         0 days 00:02:00
Name: meter, dtype: timedelta64[ns]

In [125]:
results_1200['meter'].at[(2016, 2)]

Timedelta('0 days 00:05:00')

In [119]:
meter_1200_diffs.at[(2013, 12)]

Timedelta('-1 days +23:58:00')

System 1200, Inverter:
Before 2011-11?, 15 minutes,

After 2011-12?, before 2023-11, 5 minutes,

After 2023-12, 3 minutes common (asides from the meter data, probably)

System 1200, meter:
Again, Before 2013-11, 5 minutes,

2013-12? -- 2016-01? , 3 minutes,

After 2016-02?, 5 minutes.

System 1201: 
Before 2016/01?, 3 minutes

After 2016/02?, 5 minutes

In [126]:
results_1202 = monthly_changes(
        system_id=1202, systems_cleaned=systems_cleaned,
        change_years=change_years
    )

In [11]:
inv_1202 = results_1202['inverter'].dropna()
inv_1202_diffs = inv_1202.diff()
inv_1202_diffs[~(inv_1202_diffs == timedelta(days=0))]

NameError: name 'results_1202' is not defined

In [129]:
inv_1202.head()

year  month
2010  12       0 days 00:15:00
2011  1        0 days 00:03:00
      2        0 days 00:03:00
      3        0 days 00:03:00
      4        0 days 00:03:00
Name: inverter, dtype: object

In [128]:
met_1202 = results_1202['meter'].dropna()
met_1202_diffs = met_1202.diff()
met_1202_diffs[~(met_1202_diffs == timedelta(days=0))]

year  month
2011  1                   NaT
2016  2       0 days 00:02:00
Name: meter, dtype: timedelta64[ns]

In [132]:
met_1202.loc[(2016, 2)]

Timedelta('0 days 00:05:00')

System 1202, Inverter.
2010-12? [no before] is 15 minutes of upgrade

2011-01? and after is 3 minutes of upgrade

System 1202, Meter
2016-01? and before is 3 minutes

2016-02? and after is 5 minutes

System 1203, Inverter and Meter:
Before 2016/01?, 3-minutes

2016/02? - 2018/07?, 5-minutes

2018/08? onwards, 1-second

Systems 1418-1420:

Before 2017/03?, 3-minute increment

After 2017/04?, 15-minute increment

#### Systems requiring further review:
System 51, 2011 changeover (10 or 11)?

System 1199, 2011 changeover (6 or 7)?

System 1200, Inverter, 2011 changeover (11-12), 2013 changeover (11-12)

System 1201, 2016 changeover (1 or 2)?

System 1202, Inverter, 2010-12 to 2011-1 changeover

System 1202, Meter, 2016-01 to 2016-02 changeover.

System 1203, 2016 changeover (1 or 2?), 2018 changeover (7 or 8?)

Systems 1418-1420, 2017 changeover (3 or 4?)

In [71]:
def daily_changes(
    system_id: int,
    systems_cleaned: pd.DataFrame,
    change_years: dict,
):
    # initialize data frame
    # know all data is in 1994 - 2023 range
    my_change_years = [int(year) for year in change_years[system_id]]
    num_days = len(my_change_years) * 12 * 31
    output_df = pd.DataFrame(
        data = np.full((num_days, 3), fill_value=pd.NA),
        index = pd.MultiIndex.from_product(
            [my_change_years, range(1, 13), range(1, 32)]
        ),
        columns = ['inverter', 'meter', 'unknown'],
        dtype='object'
    )
    output_df.index.names = ['year', 'month', 'day']
    # set target
    test_folder_str = f'../../../../data_ds_project/testing_yearly_parquet/{system_id}/'
    test_folder = Path(test_folder_str)
    # columns invariant among years accessed, so can just grab a year with no data to grab column names
    empty_year_pq = pq.ParquetDataset(
        test_folder,
        filters=[('year', '==', 1990)]
    )
    empty_year_df = empty_year_pq.read().to_pandas()

    # grab column names
    col_names = empty_year_df.columns
    pow_col_names = col_names[col_names.str.contains('ac_power_kW')]
    for year in my_change_years:
        current_year_pq = pq.ParquetDataset(
            test_folder,
            filters=[
                ('year', '==', year)
            ]            
        )
        current_year_df = current_year_pq.read().to_pandas()
        for pow_col_name in pow_col_names:
            pow_selection = current_year_df[['time', pow_col_name]]
            # drop if no data in the column
            pow_selection = pow_selection.dropna(axis=1, how='all')
            if (pow_selection.shape[1] > 1) and (pow_selection.shape[0] > 10):
                pow_selection.loc[:, 'month'] = pow_selection['time'].dt.month
                pow_selection.loc[:, 'day'] = pow_selection['time'].dt.day
                for month in sorted(pow_selection['month'].unique()):
                    month_selection = pow_selection[pow_selection['month'] == month]
                    clean_month = int(month)
                    if (month_selection.shape[0] > 10):
                        for day in sorted(month_selection['day'].unique()):
                            day_selection = month_selection[month_selection['day'] == day]
                            clean_day = int(day)
                            if day_selection.shape[0] > 2:
                                day_selection.loc[:, 'delta_t'] = day_selection['time'].diff()
                                day_selection_common_delta = day_selection['delta_t'].value_counts().index[0]
                                if 'inv' in pow_col_name:
                                    output_df.at[(year, clean_month, clean_day), 'inverter'] = day_selection_common_delta
                                elif 'met' in pow_col_name:
                                    output_df.at[(year, clean_month, clean_day), 'meter'] = day_selection_common_delta
                                else:
                                    output_df.at[(year, clean_month, clean_day), 'unknown'] = day_selection_common_delta
    # drop irrelevant columns
    output_df = output_df.dropna(axis=1, how='all')
    # drop irrelevant rows
    output_df = output_df.dropna(axis=0, how='all')
    # don't bother with empty dataframes
    if output_df.shape[0] == 0:
        return None
    else:
        return output_df

In [141]:
for system_id in non_standard_systems:
    print(system_id)
    results_df = daily_changes(
        system_id=system_id, systems_cleaned=systems_cleaned,
        change_years=change_years
    )
    for col in results_df.columns:
        local_data = results_df[col].dropna()
        print(col)
        start_day = local_data.index[0]
        print(f'Start: {start_day}, interval {local_data[start_day]}')
        for j in range(len(local_data.index) - 1):
            this_day = local_data.index[j]
            next_day = local_data.index[j + 1]
            if (
                abs(local_data[this_day] - local_data[next_day])
                > timedelta(seconds = 5)
            ):
                print(f'{this_day}, {local_data[this_day]}')
                print(f'{next_day}, {local_data[next_day]}')
        end_day = local_data.index[-1]
        print(f'End: {end_day}, interval {local_data[end_day]}')
        print('')

4
unknown
Start: (np.int64(2009), 1, 1), interval 0 days 00:15:00
(np.int64(2009), 6, 3), 0 days 00:15:00
(np.int64(2010), 1, 22), 0 days 00:01:00
End: (np.int64(2010), 12, 31), interval 0 days 00:01:00

10
unknown
Start: (np.int64(2000), 5, 10), interval 0 days 00:01:00
(np.int64(2000), 5, 10), 0 days 00:01:00
(np.int64(2006), 1, 9), 0 days 00:15:00
(np.int64(2009), 4, 21), 0 days 00:15:00
(np.int64(2010), 1, 10), 0 days 00:01:00
End: (np.int64(2010), 12, 31), interval 0 days 00:01:00

50
unknown
Start: (np.int64(2009), 1, 1), interval 0 days 00:15:00
(np.int64(2009), 4, 27), 0 days 00:15:00
(np.int64(2011), 4, 14), 0 days 00:01:00
End: (np.int64(2011), 12, 31), interval 0 days 00:01:00

51
unknown
Start: (np.int64(2009), 1, 1), interval 0 days 00:15:00
(np.int64(2009), 4, 22), 0 days 00:15:00
(np.int64(2011), 4, 20), 0 days 00:01:00
(np.int64(2011), 11, 1), 0 days 00:01:00
(np.int64(2011), 11, 2), 0 days 00:15:00
(np.int64(2015), 3, 9), 0 days 00:15:00
(np.int64(2015), 3, 10), 0 days

Results:

System 4: Gap in data 2009-06-03 (15 min) to 2010-01-22 (1 min)

System 10: Gap in data 2009-04-21 (15 min) to 2010-01-10 (1 min)

System 50: Gap in data 2009-04-27 (15 min) to 2011-04-14 (1 min)

System 51:

Gap in data 2009-04-22 (15 min) to 2011-04-20 (1 min)

Jump from 1 min to 15 min between 2011-11-01 and 2011-11-02 somewhere

Jump from 15 min to 1 min from 2015-03-09 to 2015-03-10 somewhere; remains 1-min until end 2015-03-21

System 1199, *Inverter*
Time-delta *approximately* 15 minutes to start (shifts of > 5 seconds shown)

Changeover to 5 min. sometime in 2011-07-21 to 2011-07-22

System 1200, *Inverter*
Time-delta *approximately* 15 minutes to start

Change to 5 min. (trial) 2011/07/20-21

Change back to 15 min. 2011/07/25-26

Change to 5 minutes somewhere 2011-12-22 to 2011-12-23

Change to 3 minutes 2013-11-28 to 2013-11-29 (mostly dead data, but not done-done until December)

System 1200, *Meter*
5 min. interval start.

3-min interval switch 2013-11-28/29 (really meter start)

Back to 5 minutes 2016-01-25/26

Continues to end in 2020.

System 1201

Starts in 2015 3-minute time interval

Switch to 5-minute time interval 2016-01-25/26

Ends end of 2016 (only 2 years' data!)

System 1202, *Inverter*
Starts 15 min 2010-12-29

Switches to 3 min. 2011-01-03/04

Ends in 2014

System 1202, *Meter*
Starts 15 minutes 2011-01-01

Switches to 3 min. 2011-01-03/4

Switch to 5-minute time interval 2016-01-25/26

Ends 2020 (same as 1200, really.)

System 1203
3-min. in 2015-01-01

*Approx.* 5-min. starting 2016-01-25/26

Switch to 1-second increment in 2018/08/03-04

Retained to end (some part of 2018)

System 1418-1419

5 min. to start (2016)

Switch to 15 min. 2017/03/28-29


System 1420

Same, except gap on 2017/03/29 [no data before, lots of data afterwards]








In [143]:
system_id = 1308
test_folder_str = f'../../../../data_ds_project/testing_yearly_parquet/{system_id}'
test_folder = Path(test_folder_str)
test_pq = pq.ParquetDataset(test_folder)
test_df = test_pq.read().to_pandas()

In [144]:
test_df

,time,ac_power_kW,year
0,2013-01-15 12:30:00,16.84347,2013
1,2013-01-15 12:45:00,15.77015,2013
2,2013-01-15 13:00:00,17.09729,2013
3,2013-01-15 13:15:00,13.99040,2013
4,2013-01-15 13:30:00,13.82301,2013
...,...,...,...
204829,2017-03-31 08:10:00,0.21309,2017
204830,2017-03-31 08:15:00,0.21633,2017
204831,2017-03-31 08:20:00,0.20700,2017
204832,2017-03-31 08:25:00,0.32919,2017


In [145]:
test_df.loc[:, 'delta_t'] = test_df['time'].diff()

In [146]:
test_df['delta_t'].value_counts()

delta_t
0 days 00:05:00     201881
0 days 00:15:00       1510
0 days 09:55:00         61
0 days 10:00:00         55
0 days 10:10:00         51
                     ...  
25 days 11:45:00         1
14 days 00:30:00         1
7 days 22:55:00          1
32 days 13:40:00         1
26 days 13:40:00         1
Name: count, Length: 73, dtype: int64

In [152]:
test_df.iloc[1000:1005]

,time,ac_power_kW,year,delta_t
1000,2013-02-06 12:30:00,9.61939,2013,0 days 00:15:00
1001,2013-02-06 12:45:00,10.10268,2013,0 days 00:15:00
1002,2013-02-06 13:00:00,17.19072,2013,0 days 00:15:00
1003,2013-02-06 13:15:00,10.83469,2013,0 days 00:15:00
1004,2013-02-06 13:30:00,12.18206,2013,0 days 00:15:00


In [153]:
test_df.iloc[2000:2005]

,time,ac_power_kW,year,delta_t
2000,2013-02-22 18:05:00,1.01217,2013,0 days 00:05:00
2001,2013-02-22 18:10:00,0.51900,2013,0 days 00:05:00
2002,2013-02-22 18:15:00,0.23819,2013,0 days 00:05:00
2003,2013-02-22 18:20:00,0.06297,2013,0 days 00:05:00
2004,2013-02-22 18:25:00,0.00263,2013,0 days 00:05:00


Aha! System 1308 changes in the middle of 2013-02.  That's why we didn't see it before.  

In [159]:
test_df.iloc[1530:1540]

,time,ac_power_kW,year,delta_t
1530,2013-02-17 18:00:00,1.50012,2013,0 days 00:15:00
1531,2013-02-17 18:15:00,0.21380,2013,0 days 00:15:00
1532,2013-02-17 18:30:00,0.00687,2013,0 days 00:15:00
1533,2013-02-17 18:45:00,0.00000,2013,0 days 00:15:00
1534,2013-02-18 17:55:00,NaN,2013,0 days 23:10:00
1535,2013-02-18 18:00:00,NaN,2013,0 days 00:05:00
1536,2013-02-18 18:05:00,NaN,2013,0 days 00:05:00
1537,2013-02-18 18:10:00,NaN,2013,0 days 00:05:00
1538,2013-02-18 18:15:00,NaN,2013,0 days 00:05:00
1539,2013-02-18 18:20:00,NaN,2013,0 days 00:05:00


In [171]:
test_df.iloc[1825:1835]

,time,ac_power_kW,year,delta_t
1825,2013-02-21 15:05:00,NaN,2013,0 days 00:05:00
1826,2013-02-21 15:10:00,NaN,2013,0 days 00:05:00
1827,2013-02-21 15:15:00,NaN,2013,0 days 00:05:00
1828,2013-02-21 15:20:00,NaN,2013,0 days 00:05:00
1829,2013-02-21 15:40:00,13.44421,2013,0 days 00:20:00
1830,2013-02-21 15:50:00,12.21946,2013,0 days 00:10:00
1831,2013-02-21 15:55:00,12.04312,2013,0 days 00:05:00
1832,2013-02-21 16:00:00,12.28889,2013,0 days 00:05:00
1833,2013-02-21 16:05:00,11.68009,2013,0 days 00:05:00
1834,2013-02-21 16:10:00,11.17940,2013,0 days 00:05:00


In effect, an outage after 2013-02-17 6:45 pm to 2013-02-21 3:40 pm to make the switch!

Unfortunately, this means that we have to be a little more careful and do day-by-day analysis of all the terms.

In [18]:
def daily_changes_no_oracle(
    system_id: int
):
    # initialize data frame
    # know all data is in 1994 - 2023 range
    num_years = 2023-1994 + 1
    num_days = num_years * 12 * 31
    output_df = pd.DataFrame(
        data = np.full((num_days, 3), fill_value=pd.NA),
        index = pd.MultiIndex.from_product(
            [range(1994, 2024), range(1, 13), range(1, 32)]
        ),
        columns = ['inverter', 'meter', 'unknown'],
        dtype='object'
    )
    output_df.index.names = ['year', 'month', 'day']
    # set target
    test_folder_str = f'../../../../data_ds_project/testing_yearly_parquet/{system_id}/'
    test_folder = Path(test_folder_str)
    # columns invariant among years accessed, so can just grab a year with no data to grab column names
    empty_year_pq = pq.ParquetDataset(
        test_folder,
        filters=[('year', '==', 1990)]
    )
    empty_year_df = empty_year_pq.read().to_pandas()

    # grab column names
    col_names = empty_year_df.columns
    pow_col_names = col_names[col_names.str.contains('ac_power_kW')]
    for year in range(1994, 2024):
        current_year_pq = pq.ParquetDataset(
            test_folder,
            filters=[
                ('year', '==', year)
            ]            
        )
        current_year_df = current_year_pq.read().to_pandas()
        for pow_col_name in pow_col_names:
            pow_selection = current_year_df[['time', pow_col_name]]
            # drop if no data in the column
            pow_selection = pow_selection.dropna(axis=1, how='all')
            if (pow_selection.shape[1] > 1) and (pow_selection.shape[0] > 10):
                pow_selection.loc[:, 'month'] = pow_selection['time'].dt.month
                pow_selection.loc[:, 'day'] = pow_selection['time'].dt.day
                for month in sorted(pow_selection['month'].unique()):
                    month_selection = pow_selection[pow_selection['month'] == month]
                    clean_month = int(month)
                    if (month_selection.shape[0] > 10):
                        for day in sorted(month_selection['day'].unique()):
                            day_selection = month_selection[month_selection['day'] == day]
                            clean_day = int(day)
                            if day_selection.shape[0] > 2:  # need any nonzero data, I'm afraid
                                day_selection.loc[:, 'delta_t'] = day_selection['time'].diff()
                                day_selection_common_delta = day_selection['delta_t'].value_counts().index[0]
                                if 'inv' in pow_col_name:
                                    output_df.at[(year, clean_month, clean_day), 'inverter'] = day_selection_common_delta
                                elif 'met' in pow_col_name:
                                    output_df.at[(year, clean_month, clean_day), 'meter'] = day_selection_common_delta
                                else:
                                    output_df.at[(year, clean_month, clean_day), 'unknown'] = day_selection_common_delta
                            elif day_selection.shape[0] == 2:  # error message
                                if 'inv' in pow_col_name:
                                    output_df.at[(year, clean_month, clean_day), 'inverter'] = timedelta(hours=11, minutes=59)
                                elif 'met' in pow_col_name:
                                    output_df.at[(year, clean_month, clean_day), 'meter'] = timedelta(hours=11, minutes=59)
                                else:
                                    output_df.at[(year, clean_month, clean_day), 'unknown'] = timedelta(hours=11, minutes=59)
                            elif day_selection.shape[0] == 1:
                                if 'inv' in pow_col_name:
                                    output_df.at[(year, clean_month, clean_day), 'inverter'] = timedelta(hours=23, minutes=59)
                                elif 'met' in pow_col_name:
                                    output_df.at[(year, clean_month, clean_day), 'meter'] = timedelta(hours=23, minutes=59)
                                else:
                                    output_df.at[(year, clean_month, clean_day), 'unknown'] = timedelta(hours=23, minutes=59)
    # drop irrelevant columns
    output_df = output_df.dropna(axis=1, how='all')
    # drop irrelevant rows
    output_df = output_df.dropna(axis=0, how='all')
    # don't bother with empty dataframes
    if output_df.shape[0] == 0:
        return None
    else:
        return output_df

In [7]:
def extract_important_daily_changes(
    system_id: int,
    na_drop_type: str = 'leading_trailing',
):
    out_list = []
    my_daily_terms = daily_changes_no_oracle(system_id=system_id)
    for col in my_daily_terms.columns:
        col_forwards = f'{col}_forward_difference'
        col_backwards = f'{col}_backward_difference'
        my_daily_terms.loc[:, col_forwards] = my_daily_terms[col].diff(periods=-1)
        my_daily_terms.loc[:, col_backwards] = my_daily_terms[col].diff(periods=1)
        # grab relevant columns and drop_na
        my_col_relevant = my_daily_terms[[col, col_forwards, col_backwards]].copy(deep=True)
        # first and last rows will always have NaT terms, so grab NaT rows
        # and any terms with large differences
        my_col_relevant = my_daily_terms[
            my_daily_terms[col_forwards].isna()
            | my_daily_terms[col_backwards].isna()
            | ((my_daily_terms[col_forwards] - my_daily_terms[col_backwards]) > timedelta(seconds=15))
            | ((my_daily_terms[col_backwards] - my_daily_terms[col_forwards]) > timedelta(seconds=15))
        ]
        return_column = my_col_relevant[col]
        # now drop NA's as required
        if na_drop_type == 'all':
            return_column = return_column.dropna()
        elif na_drop_type == 'leading_trailing':
            first_ind = return_column.first_valid_index()
            last_ind = return_column.last_valid_index()
            return_column = return_column.loc[first_ind:last_ind]
        out_list.append(return_column)
    return out_list

Need a variant for general dfs with 'date' columns

In [9]:
def daily_changes_from_df(
    simple_df: pd.DataFrame
):
    time_only_df = simple_df[['time', 'date']].copy(deep=True)
    dates_collection = time_only_df['date'].unique()
    output_ser = pd.Series(
        data = np.full(len(dates_collection), pd.NA),
        index = dates_collection,
        name = 'daily_common_timestep'
    )
    for given_date in dates_collection:
        date_selection = time_only_df[time_only_df['date'] == given_date]
        num_data_points = date_selection.shape[0]
        if num_data_points >= 10:  # now only want days with good data
            date_selection.loc[:, 'delta_t'] = date_selection['time'].diff()
            date_common_diff = date_selection['delta_t'].value_counts().index[0]
            output_ser.at[given_date] = date_common_diff
    output_ser = output_ser.dropna()  # drop definitely-bad points
    return output_ser

In [10]:
def extract_daily_changes_given_df(
    simple_df: pd.DataFrame,
    na_drop_type: str = 'leading_trailing',
):
    simple_df = simple_df[['time']].copy(deep=True)
    col_forwards = 'time_forward_difference'
    col_backwards = 'time_backward_difference'
    simple_df.loc[:, col_forwards] = simple_df['time'].diff(periods=-1)
    simple_df.loc[:, col_backwards] = simple_df['time'].diff(periods=1)
     # first and last rows will always have NaT terms, so grab NaT rows
    # and any terms with large differences
    my_col_relevant = simple_df[
        simple_df[col_forwards].isna()
        | simple_df[col_backwards].isna()
        | ((simple_df[col_forwards] - simple_df[col_backwards]) > timedelta(seconds=15))
        | ((simple_df[col_backwards] - simple_df[col_forwards]) > timedelta(seconds=15))
    ]
    return_column = my_col_relevant['time']
    # now drop NA's as required
    if na_drop_type == 'all':
        return_column = return_column.dropna()
    elif na_drop_type == 'leading_trailing':
        first_ind = return_column.first_valid_index()
        last_ind = return_column.last_valid_index()
        return_column = return_column.loc[first_ind:last_ind]
    return return_column

In [205]:
daily_changes_1200 = extract_important_daily_changes(system_id=1200, na_drop_type='all')
for df in daily_changes_1200:
    print(df)

year  month  day
2010  10     3      0 days 00:15:00
2011  7      20     0 days 00:15:00
             21     0 days 00:05:00
             25     0 days 00:05:00
             26     0 days 00:15:00
      12     22     0 days 00:15:00
             23     0 days 00:05:00
2013  11     28     0 days 00:05:00
             29     0 days 00:03:00
      12     31     0 days 00:03:00
Name: inverter, dtype: object
year  month  day
2013  1      20     0 days 00:05:00
      11     28     0 days 00:05:00
             29     0 days 00:03:00
2016  1      25     0 days 00:03:00
             26     0 days 00:05:00
2020  7      26     0 days 00:05:00
Name: meter, dtype: object


In [203]:
daily_changes_1200[0]

year  month  day
2010  10     3      0 days 00:15:00
2011  7      20     0 days 00:15:00
             21     0 days 00:05:00
             25     0 days 00:05:00
             26     0 days 00:15:00
      12     22     0 days 00:15:00
             23     0 days 00:05:00
2013  11     28     0 days 00:05:00
             29     0 days 00:03:00
      12     31     0 days 00:03:00
Name: inverter, dtype: object

In [208]:
daily_changes_1368 = extract_important_daily_changes(system_id=1368, na_drop_type='leading_trailing')
for df in daily_changes_1368:
    print(df)

year  month  day
2013  5      29     0 days 00:15:03
      9      3      0 days 00:15:01
             5      0 days 00:14:55
2018  10     14     0 days 00:15:02
             16     0 days 00:15:02
      12     31     0 days 00:15:02
Name: inverter, dtype: object


In [68]:
sample_data_1368 = pq.ParquetDataset(
     '../../../../data_ds_project/testing_yearly_parquet/1368/',
    filters= [('year', '==', 2017)]
)
sample_1368_df = sample_data_1368.read().to_pandas()
sample_1368_df.head()

,time,ac_power_kW_inverter,year
0,2017-01-01 07:00:04,0.028,2017
1,2017-01-01 07:15:08,0.367,2017
2,2017-01-01 07:30:01,0.807,2017
3,2017-01-01 07:45:02,2.782,2017
4,2017-01-01 08:00:06,NaN,2017


In [69]:
metrics_df[metrics_df['system_id'] == 1368]

,system_id,metric_id,sensor_name,common_name,raw_units,units,calc_scale,calc_offset,calc_details,aggregation_type,source_type,source_id,comments,standard_name
991,1368,3093,dc_power,DC power,W,W,1.0,0.0,inv1_dc_power+inv2_dc_power+inv3_dc_power+inv4...,avg,NaN,NaN,,dc_power__3093
992,1368,3094,ac_power,AC power,W,W,1.0,0.0,inv1_ac_power+inv2_ac_power+inv3_ac_power+inv4...,avg,NaN,NaN,,ac_power__3094
993,1368,3099,inv1_ac_power,AC power,W,W,1.0,0.0,,avg,NaN,NaN,,inv1_ac_power__3099
994,1368,3104,inv2_ac_power,AC power,W,W,1.0,0.0,,avg,NaN,NaN,,inv2_ac_power__3104
995,1368,3109,inv3_ac_power,AC power,W,W,1.0,0.0,,avg,NaN,NaN,,inv3_ac_power__3109
996,1368,3114,inv4_ac_power,AC power,W,W,1.0,0.0,,avg,NaN,NaN,,inv4_ac_power__3114
997,1368,3119,inv5_ac_power,AC power,W,W,1.0,0.0,,avg,NaN,NaN,,inv5_ac_power__3119
998,1368,3124,inv6_ac_power,AC power,W,W,1.0,0.0,,avg,NaN,NaN,,inv6_ac_power__3124
999,1368,3176,inv1_dc_power,DC power,W,W,1.0,0.0,inv1_dc_voltage*inv1_dc_current,avg,NaN,NaN,,inv1_dc_power__3176
1000,1368,3177,inv2_dc_power,DC power,W,W,1.0,0.0,inv2_dc_voltage*inv2_dc_current,avg,NaN,NaN,,inv2_dc_power__3177


Okay, so system 1368 really does have just inverter and not meter data!

In [214]:
daily_terms_1368 = daily_changes_no_oracle(system_id=1368)
daily_terms_1368.loc[(2013, 9, 3):(2013, 9, 5)].diff()

inverter
year month day                   
2013 9     3                  NaN
           4    -1 days +23:45:00
           5      0 days 00:14:54

In [213]:
daily_terms_1368.loc[(2018, 10, 14):(2018, 10, 16)]

inverter
year month day                 
2018 10    14   0 days 00:15:02
           15   0 days 00:00:01
           16   0 days 00:15:02

In [204]:
daily_changes_1200[1]

year  month  day
2013  1      20     0 days 00:05:00
      11     28     0 days 00:05:00
             29     0 days 00:03:00
2016  1      25     0 days 00:03:00
             26     0 days 00:05:00
2020  7      26     0 days 00:05:00
Name: meter, dtype: object

In [51]:
for system_id in enough_data_parquet_power_list:
    print(f'System {system_id}')
    start_end_changes = extract_important_daily_changes(
        system_id=system_id,
        na_drop_type='leading_trailing'
    )
    for df in start_end_changes:
        print(df)
        print('')

System 4
year  month  day
2007  8      26     0 days 00:15:00
2008  4      3      0 days 00:15:00
             4      0 days 07:00:00
             5      0 days 00:30:00
             6      0 days 00:15:00
2009  6      3      0 days 00:15:00
2010  1      22     0 days 00:01:00
2023  2      28     0 days 00:01:00
Name: unknown, dtype: object

System 10
year  month  day
2000  5      3             23:59:00
             10     0 days 00:01:00
2006  1      9      0 days 00:15:00
2009  4      21     0 days 00:15:00
             22            23:59:00
                         ...       
2019  3      16     0 days 02:07:00
             17     0 days 02:50:00
             18     0 days 00:08:00
             19     0 days 00:01:00
2023  2      28     0 days 00:01:00
Name: unknown, Length: 94, dtype: object

System 33
year  month  day
2010  11     8      0 days 00:01:00
2019  2      25     0 days 00:01:00
             26            23:59:00
      3      21            23:59:00
             22     

Look at too-long-to-print systems (10, 50, 51)

In [53]:
daily_data_10 = daily_changes_no_oracle(10)
daily_changes_10, = extract_important_daily_changes(10, 'leading_trailing')


In [55]:
daily_data_10.tail()

unknown
year month day                 
2023 2     24   0 days 00:01:00
           25   0 days 00:01:00
           26   0 days 00:01:00
           27   0 days 00:01:00
           28   0 days 00:01:00

In [27]:
daily_data_10_rev = daily_data_10.reset_index()
daily_data_10_rev.loc[:, 'date'] = daily_data_10_rev.apply(
    lambda row: date(row['year'], row['month'], row['day']),
    axis=1
)

In [28]:
daily_data_10_rev.head()

,year,month,day,unknown,date
0,2000,5,10,0 days 00:01:00,2000-05-10
1,2006,1,9,0 days 00:15:00,2006-01-09
2,2006,1,10,0 days 00:15:00,2006-01-10
3,2006,1,11,0 days 00:15:00,2006-01-11
4,2006,1,12,0 days 00:15:00,2006-01-12


quickly verify no issues with 2000 data.

In [32]:
data_10_yr_2000 = pq.ParquetDataset(
    '../../../../data_ds_project/testing_yearly_parquet/10/',
    filters= [('year', '==', 2000)]
)
data_10_yr_2000_df = data_10_yr_2000.read().to_pandas()
data_10_yr_2000_df

,time,ac_power_kW,year
0,2000-04-23 09:25:00,-0.009144,2000
1,2000-05-03 00:12:00,-0.108450,2000
2,2000-05-10 08:51:00,-0.106990,2000
3,2000-05-10 08:56:00,-0.107660,2000
4,2000-05-10 09:01:00,-0.107120,2000
...,...,...,...
824,2000-05-10 23:35:00,-0.106630,2000
825,2000-08-28 03:28:00,0.000239,2000
826,2000-08-30 00:47:00,0.000288,2000
827,2000-08-31 00:23:00,0.000266,2000


In [81]:
data_10_yr_2008 = pq.ParquetDataset(
    '../../../../data_ds_project/testing_yearly_parquet/10/',
    filters= [('year', '==', 2008)]
)
data_10_yr_2008_df = data_10_yr_2008.read().to_pandas()
data_10_yr_2008_df.loc[:, 'date'] = data_10_yr_2008_df['time'].dt.date

In [83]:
data_10_yr_2008_df.loc[:, 'delta_t'] = data_10_yr_2008_df['time'].diff()

In [85]:
data_10_yr_2008_df.groupby(['date'])['delta_t'].value_counts()

date        delta_t        
2008-01-01  0 days 00:15:00    95
2008-01-02  0 days 00:15:00    96
2008-01-03  0 days 00:15:00    96
2008-01-04  0 days 00:15:00    96
2008-01-05  0 days 00:15:00    96
                               ..
2008-12-27  0 days 00:15:00    96
2008-12-28  0 days 00:15:00    96
2008-12-29  0 days 00:15:00    96
2008-12-30  0 days 00:15:00    96
2008-12-31  0 days 00:15:00    96
Name: count, Length: 368, dtype: int64

OK, so not all days included, but the only day with more than 1 data point is included.

In [20]:
daily_changes_10.head()

year  month  day
2000  5      10     0 days 00:01:00
2006  1      9      0 days 00:15:00
2009  4      21     0 days 00:15:00
2010  1      10     0 days 00:01:00
2018  12     5      0 days 00:01:00
Name: unknown, dtype: object

In [56]:
daily_data_50 = daily_changes_no_oracle(50)
daily_changes_50, = extract_important_daily_changes(50, 'leading_trailing')

In [57]:
daily_data_50.tail()

unknown
year month day                 
2023 2     24   0 days 00:01:00
           25   0 days 00:01:00
           26   0 days 00:01:00
           27   0 days 00:01:00
           28   0 days 00:01:00

In [58]:
daily_changes_50.iloc[0:10]

year  month  day
1994  5      13     0 days 00:15:00
2002  2      12     0 days 00:15:00
      3      11     0 days 00:15:00
2009  4      27     0 days 00:15:00
2011  4      14     0 days 00:01:00
2013  12     18     0 days 00:01:00
             20     0 days 00:01:00
2018  9      10     0 days 00:01:00
             11     0 days 00:09:00
             12     0 days 00:03:00
Name: unknown, dtype: object

In [59]:
daily_data_50.loc[(2002, 2, 10):(2002, 2, 14)]

unknown
year month day                 
2002 2     10   0 days 00:15:00
           11   0 days 00:15:00
           12   0 days 00:15:00

So the 12th is included because it is end-of-term!

(So no data May 2009 - Mar 2011, bad days in 2018, etc.)

In [60]:
daily_data_51 = daily_changes_no_oracle(51)
daily_changes_51, = extract_important_daily_changes(51, 'leading_trailing')

In [63]:
daily_changes_51.iloc[0:10]

year  month  day
1994  5      13     0 days 00:15:00
1995  8      29     0 days 00:15:00
      9      4      0 days 00:15:00
1996  10     15     0 days 00:15:00
      11     26     0 days 00:15:00
1997  1      6      0 days 00:15:00
             21     0 days 00:15:00
      3      11     0 days 00:15:00
             28     0 days 00:15:00
      11     5      0 days 00:15:00
Name: unknown, dtype: object

In [62]:
daily_data_51.loc[(1995, 8, 28):(1995, 9, 5)]

unknown
year month day                 
1995 8     28   0 days 00:15:00
           29   0 days 00:15:00
           30          11:59:00
     9     4    0 days 00:15:00
           5    0 days 00:15:00

In [91]:
daily_data_51.dtypes

unknown    object
dtype: object

In [65]:
daily_data_51.loc[(1996, 10, 14):(1996, 11, 27)]

unknown
year month day                 
1996 10    14   0 days 00:15:00
           15   0 days 00:15:00
           16          11:59:00
     11    26   0 days 00:15:00
           27   0 days 00:15:00

In [66]:
timedelta(days=0, seconds=15*60) - timedelta(hours=11, minutes=55)

datetime.timedelta(days=-1, seconds=44400)

In [ ]:
timedelta(hours=11, minutes=55) - timedelta(days=0, seconds=15*60)

datetime.timedelta(seconds=42000)

Idea -- use the all-data (even if bad data) to grab the daily good measures, and use the other data to grab the time-steps.  

In [11]:
def grab_close_time_intervals(row):
    possible_gen_columns = ['inverter', 'meter', 'unknown']
    for col_name in possible_gen_columns:
        try:
            row[col_name]
            right_col_name = col_name
            break
        except KeyError:
            continue
        except BaseException as e:
            raise e
    if (
        (row[right_col_name] - row['daily_common_timestep']) <= timedelta(seconds = 15)
        and (row['daily_common_timestep'] - row[right_col_name]) <= timedelta(seconds=15)
    ):
        return True
    else:
        return False
    

In [12]:
def good_days_identifier_single_year(
    input_dir,
    system_id: int,
    systems_cleaned: pd.DataFrame,
    year: int, 
    run_length: int = 14,
):
    # start by grabbing all the daily deltas with or without good data
    time_steps_daily = daily_changes_no_oracle(system_id=system_id)
    system_data_types = time_steps_daily.columns 
    if len(system_data_types) == 1:
        first_timesteps = time_steps_daily[[system_data_types[0]]]
        second_timesteps = None
    elif len(time_steps_daily) == 2:
        first_timesteps = time_steps_daily[[system_data_types[0]]]
        second_timesteps = time_steps_daily[[system_data_types[1]]]
    else:
        raise ValueError(f'Bad results from daily_changes_no_oracle({system_id})')
    # identify inverter and meter
    # if inverter exists, it's the first one
    system_data_types = []
    if 'inverter' in time_steps_daily.columns:
        system_data_types.append('inverter')
    else:
        system_data_types.append('unknown')
    if (second_timesteps is not None) and (second_timesteps.shape[0] > 0):
        system_data_types.append('meter')
    # reset time
    first_timesteps = first_timesteps.reset_index()
    first_timesteps.loc[:, 'date'] = first_timesteps.apply(
        lambda row: date(row['year'], row['month'], row['day']),
        axis = 1
    )
    first_timesteps = first_timesteps.set_index('date')
    if (second_timesteps is not None) and (second_timesteps.shape[0] > 0):
        second_timesteps = second_timesteps.reset_index()
        second_timesteps.loc[:, 'date'] = second_timesteps.apply(
            lambda row: date(row['year'], row['month'], row['day']),
            axis = 1
        )
        second_timesteps = second_timesteps.set_index('date')
    # grab bad deltas
    change_days = extract_important_daily_changes(system_id, na_drop_type='leading_trailing')
    if len(change_days) == 1:
        first_changes = change_days[0]
        second_changes = None
    elif len(change_days) == 2:
        first_changes = change_days[0]
        second_changes = change_days[1]
    # reset time
    first_changes = first_changes.reset_index()
    first_changes.loc[:, 'date'] = first_changes.apply(
        lambda row: date(row['year'], row['month'], row['day']),
        axis = 1
    )
    first_bad_dates = sorted(first_changes['date'].unique())
    if (second_changes is not None) and (second_changes.shape[0] > 0):
        second_changes = second_changes.reset_index()
        second_changes.loc[:, 'date'] = second_changes.apply(
            lambda row: date(row['year'], row['month'], row['day']),
            axis = 1
        )
        second_bad_dates = sorted(second_changes['date'].unique())
    # grab yearly data, with a month before and after (as needed)
    slice_pq = pq.ParquetDataset(
        input_dir,
        filters=[('year', '>=', year - 1),
                 ('year', '<=', year + 1)]
    )
    slice_df = slice_pq.read().to_pandas()
    # trim
    slice_df = slice_df[
        (slice_df['time'] >= datetime(year-1, 12, 1))
        & (slice_df['time'] < datetime(year + 1, 2, 1))
    ]
    # drop too-small data
    cleaned_data = standardize(
        slice_df,
        'null',
        system_id,
        systems_cleaned,
        True
    )
    if len(cleaned_data) == 1:
        first_cleaned = cleaned_data[0]
        second_cleaned = None
    elif len(cleaned_data) == 2:
        first_cleaned = cleaned_data[0]
        second_cleaned = cleaned_data[1]
    else:
        raise ValueError('Trouble with standardize(*args)')
    # grab dates
    first_cleaned.loc[:, 'date'] = first_cleaned['time'].dt.date
    first_cleaned.loc[:, 'count'] = first_cleaned.groupby(['date'])['power'].transform('count')
    if (second_cleaned is not None) and (second_changes.shape[0] > 0):
        second_cleaned.loc[:, 'date'] = second_cleaned['time'].dt.date
        second_cleaned.loc[:, 'count'] = second_cleaned.groupby(['date'])['power'].transform('count')
    # apply rule to find good times.  
    first_cleaned = _terms_check(first_cleaned, first_timesteps,
                                 first_bad_dates)
    if (second_cleaned is not None) and (second_changes.shape[0] > 0):
        second_cleaned = _terms_check(second_cleaned, second_timesteps,
                                      second_bad_dates)
        return (first_cleaned, second_cleaned)
    else:
        return (first_cleaned,)


def _terms_check(
    cleaned_data,
    general_diffs,
    bad_dates,
):
    if 'date' not in cleaned_data.columns:
        cleaned_data.loc[:, 'date'] = cleaned_data['time'].dt.date
    if 'count' not in cleaned_data.columns:
        cleaned_data.loc[:, 'count'] = cleaned_data.groupby(['date'])['power'].transform('count')
    diffs_col_name = general_diffs.columns[-1]
    deltas_of_cleaned = daily_changes_from_df(cleaned_data)
    compare_timesteps = pd.merge(general_diffs, deltas_of_cleaned,
                                 how='right',
                                 left_index=True, right_index=True,
                                 suffixes=['_general', '_cleaned'])
    compare_timesteps.loc[:, 'matched'] = compare_timesteps.apply(
        grab_close_time_intervals,
        axis=1
    )
    compare_timesteps = compare_timesteps[compare_timesteps['matched']]
    compare_timesteps_dates = compare_timesteps.index.unique()
    cleaned_data = cleaned_data[cleaned_data['date'].isin(compare_timesteps_dates)]
    cleaned_data = cleaned_data[~cleaned_data['date'].isin(bad_dates)]
    min_date = cleaned_data['date'].min()
    max_date = cleaned_data['date'].max()
    # wait, have to pro-rate counts by period of goodness.  
    num_bad_dates = len(bad_dates)
    # if 2 bad dates, that is start/end, nothing to see here.
    if num_bad_dates >= 3:
        for j in range(num_bad_dates - 1):
            start_run = bad_dates[j]
            end_run = bad_dates[j + 1]
            # if irrelevant dates, nothing to do
            if (
                (end_run < min_date)
                or (start_run > max_date)
            ):
                continue
            run_selection = cleaned_data[
                    (cleaned_data['date'] > start_run)
                    & (cleaned_data['date'] < end_run)
                ].copy(deep=True)
            if (
                ((end_run - start_run) >= timedelta(days=4))  # 1-3 days not worth it
                and (run_selection.shape[0] > (24*3))
            ):
                # try to find a good delta
                temp_start_run = start_run + timedelta(days=1)
                if temp_start_run < min_date:
                    temp_start_run = min_date
                have_good_delta = False
                num_attempts = 0
                while not have_good_delta:
                    try: 
                        good_delta = general_diffs.at[temp_start_run, diffs_col_name]
                        have_good_delta = True
                    except KeyError:
                        temp_start_run += timedelta(days=1)
                        num_attempts += 1
                        if (num_attempts > 50) or (temp_start_run == end_run):
                            raise RuntimeError("Can't find a good delta!")
                run_selection.loc[:, 'forward_diff'] = run_selection['time'].diff(periods=-1)
                run_selection.loc[:, 'backward_diff'] = run_selection['time'].diff(periods=1)
                bad_data = run_selection[
                    (run_selection['forward_diff']!=good_delta)
                    & (run_selection['backward_diff']!=good_delta)
                ]
                cleaned_data.drop(index=bad_data.index)
            else:
                cleaned_data.drop(index=run_selection.index)
    return cleaned_data


In [18]:
my_single_year = standardize_single_year(
    '../../../../data_ds_project/testing_yearly_parquet/10/',
    'null',
    10,
    systems_cleaned,
    2008,
    True
)


In [146]:
my_single_year[0].head()

,time,power
30,2008-01-01 07:30:00,0.018872
31,2008-01-01 07:45:00,0.128630
32,2008-01-01 08:00:00,0.271460
33,2008-01-01 08:15:00,0.385700
34,2008-01-01 08:30:00,0.486040


In [151]:
daily_changes_10 = daily_changes_no_oracle(10)

In [153]:
daily_changes_10[(2008, 1, 1): (2008, 1, 10)]

unknown
year month day                 
2008 1     1    0 days 00:15:00
           2    0 days 00:15:00
           3    0 days 00:15:00
           4    0 days 00:15:00
           5    0 days 00:15:00
           6    0 days 00:15:00
           7    0 days 00:15:00
           8    0 days 00:15:00
           9    0 days 00:15:00
           10   0 days 00:15:00

In [147]:
inverter_2008 = my_single_year[0]
inverter_2008.loc[:, 'date'] = inverter_2008['time'].dt.date

In [148]:
inverter_2008.tail()

,time,power,date
35095,2008-12-31 15:15:00,0.64280,2008-12-31
35096,2008-12-31 15:30:00,0.55258,2008-12-31
35097,2008-12-31 15:45:00,0.47818,2008-12-31
35098,2008-12-31 16:00:00,0.35496,2008-12-31
35099,2008-12-31 16:15:00,0.23515,2008-12-31


In [120]:
inverter_2008['date'].value_counts()

date
2008-06-19    54
2008-06-26    54
2008-06-29    54
2008-07-04    54
2008-05-23    53
              ..
2008-08-16    21
2008-02-14    19
2008-12-04    18
2008-08-21    16
2008-11-30    14
Name: count, Length: 365, dtype: int64

In [150]:
daily_changes_from_df(inverter_2008).iloc[15:25]

2008-01-16    0 days 00:15:00
2008-01-17    0 days 00:15:00
2008-01-18    0 days 00:15:00
2008-01-19    0 days 00:15:00
2008-01-20    0 days 00:15:00
2008-01-21    0 days 00:15:00
2008-01-22    0 days 00:15:00
2008-01-23    0 days 00:15:00
2008-01-24    0 days 00:15:00
2008-01-25    0 days 00:15:00
Name: daily_common_timestep, dtype: object

In [19]:
data_10_yr_2008_trimmed, = good_days_identifier_single_year('../../../../data_ds_project/testing_yearly_parquet/10/',
                                 10,
                                 systems_cleaned,
                                 2008,
                                 14)

In [21]:
data_10_yr_2008_trimmed

,time,power,date,count
31715,2007-12-01 08:00:00,0.011483,2007-12-01,29
31717,2007-12-01 08:30:00,0.014360,2007-12-01,29
31718,2007-12-01 08:45:00,0.018034,2007-12-01,29
31719,2007-12-01 09:00:00,0.044024,2007-12-01,29
31720,2007-12-01 09:15:00,0.298090,2007-12-01,29
...,...,...,...,...
72093,2009-01-31 16:00:00,0.048027,2009-01-31,39
72094,2009-01-31 16:15:00,0.266930,2009-01-31,39
72095,2009-01-31 16:30:00,0.377440,2009-01-31,39
72096,2009-01-31 16:45:00,0.265140,2009-01-31,39


In [35]:
good_dates = sorted(data_10_yr_2008_trimmed['date'].unique())

In [40]:
good_dates[40:50]

[datetime.date(2008, 1, 24),
 datetime.date(2008, 1, 25),
 datetime.date(2008, 1, 26),
 datetime.date(2008, 1, 27),
 datetime.date(2008, 1, 28),
 datetime.date(2008, 1, 29),
 datetime.date(2008, 1, 30),
 datetime.date(2008, 1, 31),
 datetime.date(2008, 2, 1),
 datetime.date(2008, 2, 2)]

Having trouble running down errors, so let's try to break it down.

In [13]:
def extract_dated_data(
    input_dir,
    system_id: int,
    systems_cleaned: pd.DataFrame,
    year: int, 
):
    # start by grabbing all the daily deltas with or without good data
    time_steps_daily = daily_changes_no_oracle(system_id=system_id)
    system_data_types = time_steps_daily.columns 
    if len(system_data_types) == 1:
        first_timesteps = time_steps_daily[[system_data_types[0]]]
        second_timesteps = None
    elif len(time_steps_daily) == 2:
        first_timesteps = time_steps_daily[[system_data_types[0]]]
        second_timesteps = time_steps_daily[[system_data_types[1]]]
    else:
        raise ValueError(f'Bad results from daily_changes_no_oracle({system_id})')
    # identify inverter and meter
    # if inverter exists, it's the first one
    system_data_types = []
    if 'inverter' in time_steps_daily.columns:
        system_data_types.append('inverter')
    else:
        system_data_types.append('unknown')
    if (second_timesteps is not None) and (second_timesteps.shape[0] > 0):
        system_data_types.append('meter')
    # reset time
    first_timesteps = first_timesteps.reset_index()
    first_timesteps.loc[:, 'date'] = first_timesteps.apply(
        lambda row: date(row['year'], row['month'], row['day']),
        axis = 1
    )
    first_timesteps = first_timesteps.set_index('date')
    if (second_timesteps is not None) and (second_timesteps.shape[0] > 0):
        second_timesteps = second_timesteps.reset_index()
        second_timesteps.loc[:, 'date'] = second_timesteps.apply(
            lambda row: date(row['year'], row['month'], row['day']),
            axis = 1
        )
        second_timesteps = second_timesteps.set_index('date')
    # grab bad deltas
    change_days = extract_important_daily_changes(system_id, na_drop_type='leading_trailing')
    if len(change_days) == 1:
        first_changes = change_days[0]
        second_changes = None
    elif len(change_days) == 2:
        first_changes = change_days[0]
        second_changes = change_days[1]
    # reset time
    first_changes = first_changes.reset_index()
    first_changes.loc[:, 'date'] = first_changes.apply(
        lambda row: date(row['year'], row['month'], row['day']),
        axis = 1
    )
    first_bad_dates = sorted(first_changes['date'].unique())
    if (second_changes is not None) and (second_changes.shape[0] > 0):
        second_changes = second_changes.reset_index()
        second_changes.loc[:, 'date'] = second_changes.apply(
            lambda row: date(row['year'], row['month'], row['day']),
            axis = 1
        )
        second_bad_dates = sorted(second_changes['date'].unique())
    # grab yearly data, with a month before and after (as needed)
    slice_pq = pq.ParquetDataset(
        input_dir,
        filters=[('year', '>=', year - 1),
                 ('year', '<=', year + 1)]
    )
    slice_df = slice_pq.read().to_pandas()
    # trim
    slice_df = slice_df[
        (slice_df['time'] >= datetime(year-1, 12, 1))
        & (slice_df['time'] < datetime(year + 1, 2, 1))
    ]
    # drop too-small data
    cleaned_data = standardize(
        slice_df,
        'null',
        system_id,
        systems_cleaned,
        True
    )
    if len(cleaned_data) == 1:
        first_cleaned = cleaned_data[0]
        second_cleaned = None
    elif len(cleaned_data) == 2:
        first_cleaned = cleaned_data[0]
        second_cleaned = cleaned_data[1]
    else:
        raise ValueError('Trouble with standardize(*args)')
    # grab dates
    first_cleaned.loc[:, 'date'] = first_cleaned['time'].dt.date
    first_cleaned.loc[:, 'count'] = first_cleaned.groupby(['date'])['power'].transform('count')
    if (second_cleaned is not None) and (second_changes.shape[0] > 0):
        second_cleaned.loc[:, 'date'] = second_cleaned['time'].dt.date
        second_cleaned.loc[:, 'count'] = second_cleaned.groupby(['date'])['power'].transform('count')
    # set data to run
    if (second_cleaned is not None) and (second_changes.shape[0] > 0):
        return [(first_timesteps, first_changes, first_cleaned, first_bad_dates),
                (second_changes, second_timesteps, second_cleaned, second_bad_dates)]
    else:
        return [(first_timesteps, first_changes, first_cleaned, first_bad_dates)]

In [171]:
data_set_10_2008 = extract_dated_data(
    '../../../../data_ds_project/testing_yearly_parquet/10/',
    10,
    systems_cleaned,
    2008,
)
timesteps_10, changes_10, cleaned_10, bad_dates_10 = data_set_10_2008[0]

In [178]:
timesteps_10[date(2007, 12, 1):date(2007, 12, 20)]

,year,month,day,unknown
date,,,,
2007-12-01,2007,12,1,0 days 00:15:00
2007-12-02,2007,12,2,0 days 00:15:00
2007-12-03,2007,12,3,0 days 00:15:00
2007-12-04,2007,12,4,0 days 00:15:00
2007-12-05,2007,12,5,0 days 00:15:00
2007-12-12,2007,12,12,0 days 00:15:00
2007-12-13,2007,12,13,0 days 00:15:00
2007-12-14,2007,12,14,0 days 00:15:00
2007-12-15,2007,12,15,0 days 00:15:00


In [167]:
timesteps_10 = timesteps_10.reset_index()

In [168]:
timesteps_10['date'].dtype

dtype('O')

In [176]:
type(timesteps_10.index[12])

datetime.date

In [164]:
min_date = cleaned_10['date'].min()

In [165]:
test_date = min_date

In [177]:
test_date

datetime.date(2007, 12, 1)

In [180]:
timesteps_10.at[test_date, 'unknown']

Timedelta('0 days 00:15:00')

OK, better workflow

A.  Grab changes-data

B.  Cycle per years, clean them

C.  Grab the good-data days.

In [15]:
def daily_changes_reshaper(
    system_id: int
):
    # start by grabbing all the daily deltas with or without good data
    time_steps_daily = daily_changes_no_oracle(system_id=system_id)
    system_data_types = time_steps_daily.columns 
    if len(system_data_types) == 1:
        first_timesteps = time_steps_daily[[system_data_types[0]]]
        second_timesteps = None
    elif len(system_data_types) == 2:
        first_timesteps = time_steps_daily[[system_data_types[0]]]
        second_timesteps = time_steps_daily[[system_data_types[1]]]
    else:
        print(system_data_types)
        raise ValueError(f'Bad results from daily_changes_no_oracle({system_id})')
    # reset time
    first_timesteps = first_timesteps.reset_index()
    first_timesteps.loc[:, 'date'] = first_timesteps.apply(
        lambda row: date(row['year'], row['month'], row['day']),
        axis = 1
    )
    first_timesteps = first_timesteps.set_index('date')
    if (second_timesteps is not None) and (second_timesteps.shape[0] > 0):
        second_timesteps = second_timesteps.reset_index()
        second_timesteps.loc[:, 'date'] = second_timesteps.apply(
            lambda row: date(row['year'], row['month'], row['day']),
            axis = 1
        )
        second_timesteps = second_timesteps.set_index('date')
    # grab bad deltas
    change_days = extract_important_daily_changes(system_id, na_drop_type='leading_trailing')
    if len(change_days) == 1:
        first_changes = change_days[0]
        second_changes = None
    elif len(change_days) == 2:
        first_changes = change_days[0]
        second_changes = change_days[1]
    # reset time
    first_changes = first_changes.reset_index()
    first_changes.loc[:, 'date'] = first_changes.apply(
        lambda row: date(row['year'], row['month'], row['day']),
        axis = 1
    )
    first_bad_dates = sorted(first_changes['date'].unique())
    if (second_changes is not None) and (second_changes.shape[0] > 0):
        second_changes = second_changes.reset_index()
        second_changes.loc[:, 'date'] = second_changes.apply(
            lambda row: date(row['year'], row['month'], row['day']),
            axis = 1
        )
        second_bad_dates = sorted(second_changes['date'].unique())
    if (second_timesteps is not None) and (second_timesteps.shape[0] > 0):
        return [(first_timesteps, first_changes, first_bad_dates),
                (second_timesteps, second_changes, second_bad_dates)]
    else:
        return [(first_timesteps, first_changes, first_bad_dates),]


def just_over_year_data_extractor(
    input_dir,
    system_id: int,
    systems_cleaned: pd.DataFrame,
    year: int
):
    # grab yearly data, with a month before and after (as needed)
    slice_pq = pq.ParquetDataset(
        input_dir,
        filters=[('year', '>=', year - 1),
                 ('year', '<=', year + 1)]
    )
    slice_df = slice_pq.read().to_pandas()
    # trim
    slice_df = slice_df[
        (slice_df['time'] >= datetime(year-1, 12, 1))
        & (slice_df['time'] < datetime(year + 1, 2, 1))
    ]
    if slice_df.shape[0] < 10:
        return None
    else:
        # drop too-small data
        cleaned_data = standardize(
            slice_df,
            'null',
            system_id,
            systems_cleaned,
            True
        )
        if len(cleaned_data) == 1:
            first_cleaned = cleaned_data[0]
            second_cleaned = None
        elif len(cleaned_data) == 2:
            first_cleaned = cleaned_data[0]
            second_cleaned = cleaned_data[1]
        else:
            raise ValueError('Trouble with just_over_year_data_extractor(*args)')
        # grab dates and counts
        first_cleaned.loc[:, 'date'] = first_cleaned['time'].dt.date
        first_cleaned.loc[:, 'count'] = first_cleaned.groupby(['date'])['power'].transform('count')
        if (second_cleaned is not None) and (second_cleaned.shape[0] > 0):
            second_cleaned.loc[:, 'date'] = second_cleaned['time'].dt.date
            second_cleaned.loc[:, 'count'] = second_cleaned.groupby(['date'])['power'].transform('count')
        # return
        if (second_cleaned is not None) and (second_cleaned.shape[0] > 0):
            return (first_cleaned, second_cleaned)
        else:
            return (first_cleaned,)


def rolling_window_split(
    cleaned_data: pd.DataFrame,
    train_length: int,
    predict_length: int
):
    cleaned_data_dates = sorted(cleaned_data['date'].unique())
    possible_runs = []
    for starting_date in cleaned_data_dates:
        good_run = True
        for c in range(0, train_length + predict_length):
            c_days_ahead = starting_date + timedelta(days = c)
            if c_days_ahead not in cleaned_data_dates:
                good_run = False
                break
        if good_run:
        # extract the train_predict set and drop the training data 
            train_run = cleaned_data[
                (cleaned_data['date'] >= starting_date)
                & (cleaned_data['date'] < starting_date + timedelta(days=train_length))
            ]
            predict_run = cleaned_data[
                (cleaned_data['date'] >= starting_date + timedelta(days=train_length))
                & (cleaned_data['date'] < starting_date + timedelta(days=train_length + predict_length))
            ]
            possible_runs.append((train_run, predict_run))
    return possible_runs


def disjointify_splits(runs_list):
        if (runs_list is None) or (len(runs_list) == 0):
            return None
        elif len(runs_list) == 1:  # no overlaps
            return runs_list
        else: 
            clean_runs = []
            last_train_date = date(1993, 1, 1)  # data starts in 1993 
            for train_split, predict_split in runs_list:
                cur_train_start = train_split['date'].min()
                cur_train_end = train_split['date'].max()
                if len(clean_runs) == 0:
                    clean_runs.append((train_split, predict_split))
                    last_train_date = cur_train_end
                elif cur_train_start > last_train_date:
                    clean_runs.append((train_split, predict_split))
                    last_train_date = cur_train_end
                else:
                    continue  # not appended to clean runs.
        return clean_runs


def good_days_identifier(
    input_dir,
    system_id: int,
    systems_cleaned: pd.DataFrame,
    train_length: int = 14,
    predict_length: int = 1
):
    reshaped_data = daily_changes_reshaper(system_id=system_id)
    first_timesteps, first_changes, first_bad_dates = reshaped_data[0]
    if len(reshaped_data) == 2:
        second_timesteps, second_changes, second_bad_dates = reshaped_data[1]
    else:
        second_timesteps = None
        second_changes = None
        second_bad_dates = None
    first_splits = []
    second_splits = []
    for year in range(1994, 2024):
        cleaned_data = just_over_year_data_extractor(
            input_dir=input_dir,
            system_id=system_id,
            systems_cleaned=systems_cleaned,
            year=year
        )
        if cleaned_data is None:
            continue
        elif len(cleaned_data) == 1:
            first_cleaned = cleaned_data[0]
            second_cleaned = None
        elif len(cleaned_data) == 2:
            first_cleaned = cleaned_data[0]
            second_cleaned = cleaned_data[1]
        else:
            raise ValueError('Trouble with just_over_year_data_extractor(*args)')
        if (first_cleaned is not None) and (first_cleaned.shape[0] > 10):
            # debug
            print(year)
            first_cleaned = _terms_check_b(
                first_cleaned,
                first_timesteps,
                first_bad_dates
            )
            first_splits += rolling_window_split(first_cleaned, train_length, predict_length)
        if (second_cleaned is not None) and (second_cleaned.shape[0] > 10):
            second_cleaned = _terms_check_b(
                second_cleaned,
                second_timesteps,
                second_bad_dates
            )
            second_splits += rolling_window_split(second_cleaned, train_length, predict_length)
    # debug
    print(first_splits[0][0].head())
    print(first_splits[0][1].head())
    # now that I have all the data, *now* I can grab the disjoint training sets
    if len(first_splits) > 0:
        first_splits = disjointify_splits(first_splits)
    if len(second_splits) > 0:
        second_splits = disjointify_splits(second_splits)
    # return splits
    if len(second_splits) > 0:
        return (first_splits, second_splits)
    else:
        return (first_splits,)


In [58]:
good_days_identifier(
    '../../../../data_ds_project/testing_yearly_parquet/10/',
    10,
    systems_cleaned,
    14,
    1
)

2005
2006
2007
2008
2009
2010
2011
2012
2013
2014
2015
2016
2017
2018
2019
2020
2021
2022
2023
                    time    power        date  count
1391 2006-01-24 09:45:00  0.33858  2006-01-24     27
1392 2006-01-24 10:00:00  0.80753  2006-01-24     27
1393 2006-01-24 10:15:00  0.84531  2006-01-24     27
1394 2006-01-24 10:30:00  0.87099  2006-01-24     27
1395 2006-01-24 10:45:00  0.90351  2006-01-24     27
                    time     power        date  count
2723 2006-02-07 07:15:00  0.018749  2006-02-07     40
2724 2006-02-07 07:30:00  0.088987  2006-02-07     40
2725 2006-02-07 07:45:00  0.191040  2006-02-07     40
2726 2006-02-07 08:00:00  0.293240  2006-02-07     40
2727 2006-02-07 08:15:00  0.380230  2006-02-07     40


([(                    time    power        date  count
   1391 2006-01-24 09:45:00  0.33858  2006-01-24     27
   1392 2006-01-24 10:00:00  0.80753  2006-01-24     27
   1393 2006-01-24 10:15:00  0.84531  2006-01-24     27
   1394 2006-01-24 10:30:00  0.87099  2006-01-24     27
   1395 2006-01-24 10:45:00  0.90351  2006-01-24     27
   ...                  ...      ...         ...    ...
   2662 2006-02-06 16:00:00  0.55071  2006-02-06     40
   2663 2006-02-06 16:15:00  0.47164  2006-02-06     40
   2664 2006-02-06 16:30:00  0.38438  2006-02-06     40
   2665 2006-02-06 16:45:00  0.29113  2006-02-06     40
   2666 2006-02-06 17:00:00  0.14707  2006-02-06     40
   
   [519 rows x 4 columns],
                       time     power        date  count
   2723 2006-02-07 07:15:00  0.018749  2006-02-07     40
   2724 2006-02-07 07:30:00  0.088987  2006-02-07     40
   2725 2006-02-07 07:45:00  0.191040  2006-02-07     40
   2726 2006-02-07 08:00:00  0.293240  2006-02-07     40
   2727 2006

In [ ]:
just_over_year_data_extractor(
    '../../../../data_ds_project/testing_yearly_parquet/10/',
    10,
    systems_cleaned,
    2008
)

(                     time     power        date  count
 31715 2007-12-01 08:00:00  0.011483  2007-12-01     29
 31717 2007-12-01 08:30:00  0.014360  2007-12-01     29
 31718 2007-12-01 08:45:00  0.018034  2007-12-01     29
 31719 2007-12-01 09:00:00  0.044024  2007-12-01     29
 31720 2007-12-01 09:15:00  0.298090  2007-12-01     29
 ...                   ...       ...         ...    ...
 72093 2009-01-31 16:00:00  0.048027  2009-01-31     39
 72094 2009-01-31 16:15:00  0.266930  2009-01-31     39
 72095 2009-01-31 16:30:00  0.377440  2009-01-31     39
 72096 2009-01-31 16:45:00  0.265140  2009-01-31     39
 72097 2009-01-31 17:00:00  0.055423  2009-01-31     39
 
 [17004 rows x 4 columns],)

In [16]:
def _terms_check_b(
    cleaned_data,
    general_diffs,
    bad_dates,
):
    if 'date' not in cleaned_data.columns:
        cleaned_data.loc[:, 'date'] = cleaned_data['time'].dt.date
    if 'count' not in cleaned_data.columns:
        cleaned_data.loc[:, 'count'] = cleaned_data.groupby(['date'])['power'].transform('count')
    if cleaned_data.shape[0] == 0:
        raise ValueError('Bad cleaned_data input!')
    diffs_col_name = general_diffs.columns[-1]
    deltas_of_cleaned = daily_changes_from_df(cleaned_data)
    if deltas_of_cleaned is None or deltas_of_cleaned.shape[0] == 0:
        print('No data pre-merge')
        return None
    compare_timesteps = pd.merge(general_diffs, deltas_of_cleaned,
                                 how='right',
                                 left_index=True, right_index=True,
                                 suffixes=['_general', '_cleaned'])
    # check for nones
    if compare_timesteps.shape[0] == 0:
        print('No data post-merge')
        return None
    compare_timesteps.loc[:, 'matched'] = compare_timesteps.apply(
        grab_close_time_intervals,
        axis=1
    )
    compare_timesteps = compare_timesteps[compare_timesteps['matched']]
    compare_timesteps_dates = compare_timesteps.index.unique()
    cleaned_data = cleaned_data[cleaned_data['date'].isin(compare_timesteps_dates)]
    cleaned_data = cleaned_data[~cleaned_data['date'].isin(bad_dates)]
    if cleaned_data.shape[0] == 0:
        print('No data post-pruning')
        return None
    min_date = cleaned_data['date'].min()
    max_date = cleaned_data['date'].max()
    # wait, have to pro-rate counts by period of goodness.  
    num_bad_dates = len(bad_dates)
    # if 2 bad dates, that is start/end, nothing to see here.
    if num_bad_dates >= 3:
        for j in range(num_bad_dates - 1):
            start_run = bad_dates[j]
            end_run = bad_dates[j + 1]
            # if irrelevant dates, nothing to do
            try:
                my_cond = (end_run < min_date) or (start_run > max_date)
            except TypeError as e:
                print(start_run)
                print(end_run)
                print(min_date)
                print(max_date)
                raise e
            except BaseException as e:
                raise e
            if my_cond:
                continue
            run_selection = cleaned_data[
                    (cleaned_data['date'] > start_run)
                    & (cleaned_data['date'] < end_run)
                ].copy(deep=True)
            if (
                ((end_run - start_run) >= timedelta(days=4))  # 1-3 days not worth it
                and (run_selection.shape[0] > (24*3))
            ):
                # try to find a good delta
                temp_start_run = start_run + timedelta(days=1)
                if temp_start_run < min_date:
                    temp_start_run = min_date
                have_good_delta = False
                num_attempts = 0
                while not have_good_delta:
                    try: 
                        good_delta = general_diffs.at[temp_start_run, diffs_col_name]
                        have_good_delta = True
                    except KeyError:
                        temp_start_run += timedelta(days=1)
                        num_attempts += 1
                        if (num_attempts > 50) or (temp_start_run == end_run):
                            raise RuntimeError("Can't find a good delta!")
                run_selection.loc[:, 'forward_diff'] = run_selection['time'].diff(periods=-1)
                run_selection.loc[:, 'backward_diff'] = run_selection['time'].diff(periods=1)
                # first difference -- more tolerance in delta
                bad_data = run_selection[
                    ((run_selection['forward_diff'] > (good_delta + timedelta(seconds = 15)))
                     | (run_selection['forward_diff'] < (good_delta - timedelta(seconds = 15))))
                    & ((run_selection['backward_diff'] > (good_delta + timedelta(seconds = 15)))
                       | (run_selection['backward_diff'] < (good_delta -  timedelta(seconds=15))))
                ]
                cleaned_data.drop(index=bad_data.index)
            else:
                cleaned_data.drop(index=run_selection.index)
    return cleaned_data


In [52]:
rng = np.random.default_rng()
sample_data = rng.normal(loc=0, scale=1, size = (24, 2))
sample_df = pd.DataFrame(
    data=sample_data,
    columns = ['test_a', 'test_b']
)

In [53]:
sample_df[sample_df['test_b'] > 1]

,test_a,test_b
4,-0.493713,1.558418
5,1.443263,1.401303
16,-0.359864,1.325388
18,0.915491,1.429159


In [51]:
sample_data[sample_data > 0.5]

array([1.59168541, 0.99550866, 0.61692813, 0.62608879, 1.24415397,
       0.94626094, 0.50408807, 0.85811397, 0.82801835, 2.8411642 ,
       0.56488457, 1.98801356, 2.18777648, 0.91270091, 0.96666962,
       1.04395592, 0.94187419, 1.91273203, 1.17138019, 0.62777221,
       0.9931881 , 2.12969666, 1.15165183, 0.52357632, 1.24674803,
       1.35164756, 0.57278877, 1.02355957, 0.92245159, 0.52018758,
       1.38701304, 1.10112177, 1.06794687, 1.03731518, 0.72398331,
       1.52177839, 1.03814294, 1.48853508, 0.87418045, 1.12758282,
       0.91697857, 1.73157174, 1.00636349, 1.00407717, 0.52331793,
       1.02279232, 1.51070366, 1.23909855, 0.71443391, 0.86496268,
       1.72339904, 1.27776548, 0.95837276, 0.95018046, 1.22015725,
       0.63017979, 0.51681179, 1.69648868, 0.52419093, 0.73949075,
       2.72574484, 0.8312641 , 0.55190395, 0.93647154, 1.44922898,
       0.62772327, 1.31643941, 2.31736296, 1.01605373, 1.63718146,
       0.76745001, 1.34367955, 0.62695253, 1.57815885, 1.45867

(Aside: if I just want the good dates, then I don't need the overlap!)

In [17]:
def year_data_extractor(
    input_dir,
    system_id: int,
    systems_cleaned: pd.DataFrame,
    year: int
):
    # grab yearly data, with a month before and after (as needed)
    slice_pq = pq.ParquetDataset(
        input_dir,
        filters=[('year', '==', year)]
    )
    slice_df = slice_pq.read().to_pandas()

    if slice_df.shape[0] < 10:
        return None
    else:
        # drop too-small data
        cleaned_data = standardize(
            slice_df,
            'null',
            system_id,
            systems_cleaned,
            True
        )
        if len(cleaned_data) == 1:
            first_cleaned = cleaned_data[0]
            second_cleaned = None
        elif len(cleaned_data) == 2:
            first_cleaned = cleaned_data[0]
            second_cleaned = cleaned_data[1]
        else:
            raise ValueError('Trouble with just_over_year_data_extractor(*args)')
        # grab dates and counts
        first_cleaned.loc[:, 'date'] = first_cleaned['time'].dt.date
        first_cleaned.loc[:, 'count'] = first_cleaned.groupby(['date'])['power'].transform('count')
        if (second_cleaned is not None) and (second_cleaned.shape[0] > 0):
            second_cleaned.loc[:, 'date'] = second_cleaned['time'].dt.date
            second_cleaned.loc[:, 'count'] = second_cleaned.groupby(['date'])['power'].transform('count')
        # return
        if (second_cleaned is not None) and (second_cleaned.shape[0] > 0):
            return (first_cleaned, second_cleaned)
        else:
            return (first_cleaned,)


def good_days_only_identifier(
    input_dir,
    system_id: int,
    systems_cleaned: pd.DataFrame,
):
    reshaped_data = daily_changes_reshaper(system_id=system_id)
    first_timesteps, first_changes, first_bad_dates = reshaped_data[0]
    if len(reshaped_data) == 2:
        second_timesteps, second_changes, second_bad_dates = reshaped_data[1]
    else:
        second_timesteps = None
        second_changes = None
        second_bad_dates = None
    first_cleaned_compendium = []
    second_cleaned_compendium = []
    for year in range(1994, 2024):
        cleaned_data = year_data_extractor(
            input_dir=input_dir,
            system_id=system_id,
            systems_cleaned=systems_cleaned,
            year=year
        )
        if cleaned_data is None:
            first_cleaned = None
            second_cleaned = None
            continue
        elif len(cleaned_data) == 1:
            first_cleaned = cleaned_data[0]
            second_cleaned = None
        elif len(cleaned_data) == 2:
            first_cleaned = cleaned_data[0]
            second_cleaned = cleaned_data[1]
        else:
            raise ValueError('Trouble with just_over_year_data_extractor(*args)')
        if (first_cleaned is not None) and (first_cleaned.shape[0] > 10):
            first_cleaned = _terms_check_b(
                first_cleaned,
                first_timesteps,
                first_bad_dates
            )
            if first_cleaned is not None:
                first_cleaned_compendium.append(first_cleaned)
        if (second_cleaned is not None) and (second_cleaned.shape[0] > 10):
            second_cleaned = _terms_check_b(
                second_cleaned,
                second_timesteps,
                second_bad_dates
            )
            if second_cleaned is not None:
                second_cleaned_compendium.append(second_cleaned)
    first_cleaned_total = pd.concat(first_cleaned_compendium)
    if (second_cleaned is not None) and (second_cleaned.shape[0] > 10):
        second_cleaned_total = pd.concat(second_cleaned_compendium)
    if (second_cleaned is not None) and (second_cleaned.shape[0] > 10):
        return (first_cleaned_total, second_cleaned_total)
    else:
        return (first_cleaned_total,)


In [70]:
very_clean_day_10_data, = good_days_only_identifier(
    '../../../../data_ds_project/testing_yearly_parquet/10/',
    10,
    systems_cleaned,
)
len(very_clean_day_10_data['date'].unique())

5635

In [71]:
very_clean_day_10_data.head()

,time,power,date,count
1391,2006-01-24 09:45:00,0.33858,2006-01-24,27
1392,2006-01-24 10:00:00,0.80753,2006-01-24,27
1393,2006-01-24 10:15:00,0.84531,2006-01-24,27
1394,2006-01-24 10:30:00,0.87099,2006-01-24,27
1395,2006-01-24 10:45:00,0.90351,2006-01-24,27


Time-check

In [28]:
enough_data_parquet_power_list[10:13]

[1201, 1202, 1203]

In [33]:
good_days_only_identifier(
    '../../../../data_ds_project/testing_yearly_parquet/51/',
    51,
    systems_cleaned,
)

No data pre-merge


(                     time     power        date  count
 54    1994-05-14 05:30:00  0.070800  1994-05-14     50
 55    1994-05-14 05:45:00  0.283900  1994-05-14     50
 56    1994-05-14 06:00:00  0.217800  1994-05-14     50
 57    1994-05-14 06:15:00  0.950000  1994-05-14     50
 58    1994-05-14 06:30:00  1.415000  1994-05-14     50
 ...                   ...       ...         ...    ...
 83128 2023-02-27 17:28:00  0.064042  2023-02-27    648
 83129 2023-02-27 17:29:00  0.064874  2023-02-27    648
 83130 2023-02-27 17:30:00  0.069672  2023-02-27    648
 83131 2023-02-27 17:31:00  0.063884  2023-02-27    648
 83132 2023-02-27 17:32:00  0.064043  2023-02-27    648
 
 [1823954 rows x 4 columns],)

In [40]:
for system_id in tqdm(enough_data_parquet_power_list):
    major_dir = f'../../../../data_ds_project/testing_yearly_parquet/{system_id}/'
    good_days_only_identifier(
        major_dir,
        system_id,
        systems_cleaned
    )

  9%|▊         | 7/82 [18:20<2:49:50, 135.87s/it]

No data pre-merge


 13%|█▎        | 11/82 [20:35<1:02:47, 53.07s/it]

No data post-pruning
No data post-pruning
No data post-pruning


 35%|███▌      | 29/82 [31:31<07:40,  8.69s/it]   

...

 56%|█████▌    | 46/82 [33:31<04:23,  7.31s/it]

No data post-pruning


 95%|█████████▌| 78/82 [59:55<01:06, 16.70s/it]   

No data post-pruning


100%|██████████| 82/82 [1:02:29<00:00, 45.73s/it]


In [37]:
daily_changes_1200 = daily_changes_no_oracle(1200)

In [39]:
daily_changes_1200.columns

Index(['inverter', 'meter'], dtype='str')

In [38]:
daily_changes_1200

inverter            meter
year month day                                  
2010 10    3    0 days 00:15:00             <NA>
           4    0 days 00:15:00             <NA>
           5    0 days 00:15:00             <NA>
           6    0 days 00:15:00             <NA>
           7    0 days 00:15:00             <NA>
...                         ...              ...
2020 7     22              <NA>  0 days 00:05:00
           23              <NA>  0 days 00:05:00
           24              <NA>  0 days 00:05:00
           25              <NA>  0 days 00:05:00
           26              <NA>  0 days 00:05:00

[3354 rows x 2 columns]

In [26]:
good_days_only_identifier(
    '../../../../data_ds_project/testing_yearly_parquet/1200/',
    1200,
    systems_cleaned,
)

(                     time     power        date  count
 29    2010-10-04 08:15:44  0.573328  2010-10-04     40
 30    2010-10-04 08:30:44  0.915454  2010-10-04     40
 31    2010-10-04 08:45:44  0.954552  2010-10-04     40
 32    2010-10-04 09:00:44  1.131823  2010-10-04     40
 33    2010-10-04 09:15:44  1.540468  2010-10-04     40
 ...                   ...       ...         ...    ...
 34659 2013-11-27 15:30:33  1.004817  2013-11-27     90
 34660 2013-11-27 15:35:33  1.098343  2013-11-27     90
 34661 2013-11-27 15:40:33  0.979800  2013-11-27     90
 34662 2013-11-27 15:45:33  0.914381  2013-11-27     90
 34663 2013-11-27 15:50:33  0.760205  2013-11-27     90
 
 [80186 rows x 4 columns],)

In [39]:
good_days_only_identifier(
    '../../../../data_ds_project/testing_yearly_parquet/1202/',
    1202,
    systems_cleaned,
)

No data post-pruning
No data post-pruning
No data post-pruning


(                  time     power        date  count
 18 2010-12-30 07:45:03  0.502363  2010-12-30     36
 19 2010-12-30 08:00:03  1.496045  2010-12-30     36
 20 2010-12-30 08:15:03  2.529429  2010-12-30     36
 21 2010-12-30 08:30:03  3.988605  2010-12-30     36
 22 2010-12-30 08:45:03  5.425650  2010-12-30     36
 ..                 ...       ...         ...    ...
 67 2011-01-02 14:30:03  0.755809  2011-01-02     26
 68 2011-01-02 14:45:03  0.724795  2011-01-02     26
 69 2011-01-02 15:00:03  0.882000  2011-01-02     26
 70 2011-01-02 15:15:03  1.475868  2011-01-02     26
 71 2011-01-02 15:30:03  1.576174  2011-01-02     26
 
 [132 rows x 4 columns],)